In [ ]:
import requests
from Credentials import CLIENT_ID, CLIENT_SECRET, engine_DEV_Final as engine
from BCDA_API import get_access_token


access_token = get_access_token()

response = requests.delete(
    'https://api.bcda.cms.gov/api/v3/jobs/177664',
    headers={
        "Accept": "application/fhir+json",
        "Authorization": f"Bearer {access_token}",
    }
)


Authenticating...
Token HTTP Status: 200
Access token acquired (prefix): eyJhbGciOiJSUzUxMiIs ...


In [7]:
response.headers

{'Date': 'Wed, 01 Jul 2026 15:50:29 GMT', 'Content-Length': '0', 'Connection': 'keep-alive', 'Cache-Control': 'no-cache; no-store; must-revalidate; max-age=0', 'Pragma': 'no-cache', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'X-Content-Type-Options': 'nosniff', 'X-Progress': 'In Progress (0%)'}

In [96]:
import polars as pl
from pathlib import Path
import pandas as pd
import duckdb

con = duckdb.connect()

df_eob = con.execute("select * from read_ndjson_auto('C:\\BCDA_V3\\Data\\ExplanationOfBenefit_v3_0137_20260706193338.ndjson', sample_size=-1, filename=true)").pl()
#df_patient = con.execute("select * from read_ndjson_auto('C:\\BCDA_V3\\Data\\Patient*.ndjson', sample_size=-1,union_by_name=true, filename=true)").pl()
#df_coverage = con.execute("select * from read_ndjson_auto('C:\\BCDA_V3\\Data\\Coverage*.ndjson', sample_size=-1,union_by_name=true, filename=true)").pl()


In [21]:
def flatten(df: pl.DataFrame, col: str) -> pl.DataFrame:
    return df.explode(col).unnest(col)
df_patient_race =(
df_patient
.select(['id', 'extension'])
.pipe(flatten, 'extension')
.select(['id', 'extension'])
.pipe(flatten, 'extension')
.select(['id','valueString'])
.filter(pl.col('valueString').is_not_null())
)

patient_base = (
    df_patient
    .select([
        "id",
        "name",
        "gender",
        "birthDate",
        "address",
        "deceasedDateTime",
        'filename',
        'meta',
        'link'
    ])
    .explode("address")
    .unnest("address")
    .explode("name")
    .unnest("name")
    .explode('link')
    .unnest('link')
    .with_columns(
        pl.col("given").list.join(", "),
        pl.col('meta').struct.field('lastUpdated'),
        pl.col('other').struct.field('display').alias('id_link')
    ).join(df_patient_race.select(['id', 'valueString']), on= 'id', how= 'left')
)

patient_base_sorted = (
    patient_base
    .with_columns(
    pl.col('valueString').alias('race'),
    pl.col('filename').str.split('\\').list.get(-1).alias('filename')
    )
    .select(
        "id",
        'id_link',
        'family',
        'given',
        "gender",
        'race',
        "birthDate",
        "postalCode",
        'state',
        "deceasedDateTime",
        'lastUpdated',
        'filename'
))

patient_base_sorted

id,id_link,family,given,gender,race,birthDate,postalCode,state,deceasedDateTime,lastUpdated,filename
str,str,str,str,str,str,date,str,str,date,datetime[μs],str
"""146341947""",null,"""DOSS""","""WILLIAM, C""","""male""","""White""",1992-02-15,"""75670""","""TX""",null,2026-05-24 07:15:29.760,"""Patient_v3_0001_20260706193338…"
"""164737228""",null,"""THOMASON""","""LANA""","""female""","""White""",1956-01-27,"""75501""","""TX""",null,2026-06-19 07:39:14.270,"""Patient_v3_0001_20260706193338…"
"""142606117""",null,"""WOOD""","""JUDITH""","""female""","""White""",1951-07-27,"""71854""","""AR""",null,2026-06-02 09:42:15.341,"""Patient_v3_0001_20260706193338…"
"""147591073""",null,"""BARNES""","""JAMES, R""","""male""","""White""",1952-10-28,"""75503""","""TX""",null,2026-03-29 21:19:18.767,"""Patient_v3_0001_20260706193338…"
"""128045656""",null,"""MCCORMICK""","""WALTER, L""","""male""","""White""",1947-08-08,"""75572""","""TX""",null,2026-04-09 01:16:58.310,"""Patient_v3_0001_20260706193338…"
…,…,…,…,…,…,…,…,…,…,…,…
"""113605516""",null,"""HILLIS""","""JUDA""","""female""","""White""",1943-06-12,"""75567""","""TX""",null,2026-06-02 09:42:15.341,"""Patient_v3_0005_20260706193338…"
"""168261951""",null,"""WILHOIT""","""JOHNNY, M""","""male""","""White""",1958-01-24,"""75601""","""TX""",null,2026-05-24 09:42:08.434,"""Patient_v3_0005_20260706193338…"
"""167546408""",null,"""JORDAN""","""CHESLEY, D""","""male""","""White""",1977-12-27,"""75501""","""TX""",null,2026-03-29 21:29:42.884,"""Patient_v3_0005_20260706193338…"


In [12]:
df_patient

address,birthDate,communication,extension,gender,id,identifier,meta,name,resourceType,deceasedDateTime,link,filename
list[struct[2]],date,list[struct[1]],list[struct[3]],str,str,list[struct[4]],struct[2],list[struct[2]],str,date,list[struct[2]],str
"[{""75604"",""TX""}]",1971-03-02,"[{{[{""unknown"",""urn:ietf:bcp:47""}]}}]","[{""http://hl7.org/fhir/us/core/StructureDefinition/us-core-sex"",""248152002"",null}, {""http://hl7.org/fhir/us/core/StructureDefinition/us-core-race"",null,[{""ombCategory"",{""2106-3"",""White"",""urn:oid:2.16.840.1.113883.6.238""},null}, {""text"",null,""White""}]}]","""female""","""176077134""","[{{2024-08-13 00:00:00,null},""http://hl7.org/fhir/sid/us-mbi"",{[{""MB"",""http://terminology.hl7.org/CodeSystem/v2-0203""}]},""4T94HG4AW74""}]","{2026-03-09 03:34:52.931,[""http://hl7.org/fhir/us/carin-bb/StructureDefinition/C4BB-Patient|2.2.0"", ""http://hl7.org/fhir/us/core/StructureDefinition/us-core-patient|7.0.0""]}","[{""SPRUCE"",[""REBECCA"", ""J""]}]","""Patient""",null,null,"""C:\BCDA_V3\Data\Patient_v3_000…"
"[{""75571"",""TX""}]",1942-02-28,"[{{[{""unknown"",""urn:ietf:bcp:47""}]}}]","[{""http://hl7.org/fhir/us/core/StructureDefinition/us-core-sex"",""248153007"",null}, {""http://hl7.org/fhir/us/core/StructureDefinition/us-core-race"",null,[{""ombCategory"",{""2106-3"",""White"",""urn:oid:2.16.840.1.113883.6.238""},null}, {""text"",null,""White""}]}]","""male""","""109664215""","[{{2017-05-20 00:00:00,null},""http://hl7.org/fhir/sid/us-mbi"",{[{""MB"",""http://terminology.hl7.org/CodeSystem/v2-0203""}]},""2RP0GP3DU25""}]","{2026-03-12 04:32:52.087,[""http://hl7.org/fhir/us/carin-bb/StructureDefinition/C4BB-Patient|2.2.0"", ""http://hl7.org/fhir/us/core/StructureDefinition/us-core-patient|7.0.0""]}","[{""WARD"",[""WALTER"", ""T""]}]","""Patient""",null,null,"""C:\BCDA_V3\Data\Patient_v3_000…"
"[{""75567"",""TX""}]",1961-04-18,"[{{[{""unknown"",""urn:ietf:bcp:47""}]}}]","[{""http://hl7.org/fhir/us/core/StructureDefinition/us-core-sex"",""248152002"",null}, {""http://hl7.org/fhir/us/core/StructureDefinition/us-core-race"",null,[{""ombCategory"",{""2106-3"",""White"",""urn:oid:2.16.840.1.113883.6.238""},null}, {""text"",null,""White""}]}]","""female""","""181503373""","[{{2025-11-24 00:00:00,null},""http://hl7.org/fhir/sid/us-mbi"",{[{""MB"",""http://terminology.hl7.org/CodeSystem/v2-0203""}]},""5RU5PG9CA83""}]","{2026-03-09 04:13:27.353,[""http://hl7.org/fhir/us/carin-bb/StructureDefinition/C4BB-Patient|2.2.0"", ""http://hl7.org/fhir/us/core/StructureDefinition/us-core-patient|7.0.0""]}","[{""MARTIN"",[""DONNA"", ""M""]}]","""Patient""",null,null,"""C:\BCDA_V3\Data\Patient_v3_000…"
"[{""75670"",""TX""}]",1992-02-15,"[{{[{""unknown"",""urn:ietf:bcp:47""}]}}]","[{""http://hl7.org/fhir/us/core/StructureDefinition/us-core-sex"",""248153007"",null}, {""http://hl7.org/fhir/us/core/StructureDefinition/us-core-race"",null,[{""ombCategory"",{""2106-3"",""White"",""urn:oid:2.16.840.1.113883.6.238""},null}, {""text"",null,""White""}]}]","""male""","""146341947""","[{{2017-05-20 00:00:00,null},""http://hl7.org/fhir/sid/us-mbi"",{[{""MB"",""http://terminology.hl7.org/CodeSystem/v2-0203""}]},""5AU9NY9RE37""}]","{2026-05-24 07:15:29.760,[""http://hl7.org/fhir/us/carin-bb/StructureDefinition/C4BB-Patient|2.2.0"", ""http://hl7.org/fhir/us/core/StructureDefinition/us-core-patient|7.0.0""]}","[{""DOSS"",[""WILLIAM"", ""C""]}]","""Patient""",null,null,"""C:\BCDA_V3\Data\Patient_v3_000…"
"[{""75691"",""TX""}]",1955-11-20,"[{{[{""unknown"",""urn:ietf:bcp:47""}]}}]","[{""http://hl7.org/fhir/us/core/StructureDefinition/us-core-sex"",""248152002"",null}, {""http://hl7.org/fhir/us/core/StructureDefinition/us-core-race"",null,[{""ombCategory"",{""2106-3"",""White"",""urn:oid:2.16.840.1.113883.6.238""},null}, {""text"",null,""White""}]}]","""female""","""159744461""","[{{2020-06-25 00:00:00,null},""http://hl7.org/fhir/sid/us-mbi"",{[{""MB"",""http://terminology.hl7.org/CodeSystem/v2-0203""}]},""4VY4QU5TF01""}]","{2026-03-12 05

In [11]:
df_patient_race =(
df_patient
.select(['id', 'extension'])
.explode('extension')
.unnest('extension')

)
df_patient_race

id,url,valueCode,extension
str,str,str,list[struct[3]]
"""176077134""","""http://hl7.org/fhir/us/core/St…","""248152002""",null
"""176077134""","""http://hl7.org/fhir/us/core/St…",null,"[{""ombCategory"",{""2106-3"",""White"",""urn:oid:2.16.840.1.113883.6.238""},null}, {""text"",null,""White""}]"
"""109664215""","""http://hl7.org/fhir/us/core/St…","""248153007""",null
"""109664215""","""http://hl7.org/fhir/us/core/St…",null,"[{""ombCategory"",{""2106-3"",""White"",""urn:oid:2.16.840.1.113883.6.238""},null}, {""text"",null,""White""}]"
"""181503373""","""http://hl7.org/fhir/us/core/St…","""248152002""",null
…,…,…,…
"""99908141""","""http://hl7.org/fhir/us/core/St…",null,"[{""ombCategory"",{""2106-3"",""White"",""urn:oid:2.16.840.1.113883.6.238""},null}, {""text"",null,""White""}]"
"""164660944""","""http://hl7.org/fhir/us/core/St…","""248153007""",null
"""164660944""","""http://hl7.org/fhir/us/core/St…",null,"[{""ombCategory"",{""2106-3"",""White"",""urn:oid:2.16.840.1.113883.6.238""},null}, {""text"",null,""White""}]"


In [53]:
df_patient_linktable = (
    df_patient
    .filter(pl.col('identifier').is_not_null())
    .select(
        'id',
        'identifier',
        'filename',
        'meta'
    )
    .explode('identifier')
    .unnest('identifier')
    .select(
        'id',
        'period',
        'value',
        'filename',
        'meta'
    )
    .unnest('period')
    .with_columns(
        pl.col('meta').struct.field('lastUpdated')
    )
    .select(
        'id',
        'start',
        'end',
        'value',
        'lastUpdated',
        'filename'
    )
)
df_patient_linktable

id,start,end,value,lastUpdated,filename
str,datetime[μs],datetime[μs],str,datetime[μs],str
"""176077134""",2024-08-13 00:00:00,null,"""4T94HG4AW74""",2026-03-09 03:34:52.931,"""C:\BCDA_V3\Data\Patient_v3_000…"
"""109664215""",2017-05-20 00:00:00,null,"""2RP0GP3DU25""",2026-03-12 04:32:52.087,"""C:\BCDA_V3\Data\Patient_v3_000…"
"""181503373""",2025-11-24 00:00:00,null,"""5RU5PG9CA83""",2026-03-09 04:13:27.353,"""C:\BCDA_V3\Data\Patient_v3_000…"
"""146341947""",2017-05-20 00:00:00,null,"""5AU9NY9RE37""",2026-05-24 07:15:29.760,"""C:\BCDA_V3\Data\Patient_v3_000…"
"""159744461""",2020-06-25 00:00:00,null,"""4VY4QU5TF01""",2026-03-12 05:04:55.361,"""C:\BCDA_V3\Data\Patient_v3_000…"
…,…,…,…,…,…
"""99908141""",2017-05-20 00:00:00,2026-04-14 00:00:00,"""8QK9N92NT31""",2026-03-11 21:09:00.092,"""C:\BCDA_V3\Data\Patient_v3_000…"
"""99908141""",2026-04-14 00:00:00,null,"""9PE5G68QM81""",2026-03-11 21:09:00.092,"""C:\BCDA_V3\Data\Patient_v3_000…"
"""164660944""",2021-10-13 00:00:00,null,"""4QA5CQ3EV84""",2026-03-12 06:12:30.355,"""C:\BCDA_V3\Data\Patient_v3_000…"


In [78]:
df_coverage_base = (
    df_coverage
    .select(
        'id',
        'beneficiary',
        'meta',
        'period',
        'payor',
        'status',
        'subscriberId',
        'filename'
    )
    .explode('payor')
    .unnest('payor')
    .unnest('period')
    .with_columns(
        pl.col('beneficiary')
        .struct.field('reference')
        .str.split('/')
        .list.get(1)
        .alias('patient_id'),
        pl.col('meta').struct.field('lastUpdated'),
    )
    .select(
        'id',
        'patient_id',
        'reference',
        'start',
        'end',
        'status',
        'subscriberId',
        'lastUpdated',
        'filename'
    )
)
df_coverage_base

id,patient_id,reference,start,end,status,subscriberId,lastUpdated,filename
str,str,str,date,date,str,str,datetime[μs],str
"""part-a-146341947""","""146341947""","""#cms-org""",2017-06-01,null,"""active""","""5AU9NY9RE37""",2026-05-24 07:15:29.760,"""C:\BCDA_V3\Data\Coverage_v3_04…"
"""part-b-146341947""","""146341947""","""#cms-org""",2017-06-01,null,"""active""","""5AU9NY9RE37""",2026-05-24 07:15:29.760,"""C:\BCDA_V3\Data\Coverage_v3_04…"
"""part-d-146341947-S5884-168""","""146341947""","""#insurer-org""",2026-07-01,null,"""active""","""5AU9NY9RE37""",2026-06-26 08:38:32.811,"""C:\BCDA_V3\Data\Coverage_v3_04…"
"""dual-146341947""","""146341947""","""#cms-org""",2021-07-01,null,"""active""","""5AU9NY9RE37""",2026-05-24 07:15:29.760,"""C:\BCDA_V3\Data\Coverage_v3_04…"
"""part-a-175267566""","""175267566""","""#cms-org""",2024-07-01,null,"""active""","""5TV2QE7PC96""",2026-06-25 10:12:47.118,"""C:\BCDA_V3\Data\Coverage_v3_04…"
…,…,…,…,…,…,…,…,…
"""part-b-161497185""","""161497185""","""#cms-org""",2020-10-01,2026-07-31,"""active""","""4AY0V77DJ35""",2026-06-26 08:38:32.769,"""C:\BCDA_V3\Data\Coverage_v3_04…"
"""part-c-161497185-H2293-014""","""161497185""","""#insurer-org""",2026-02-01,2026-07-31,"""active""","""4AY0V77DJ35""",2026-06-26 08:38:32.811,"""C:\BCDA_V3\Data\Coverage_v3_04…"
"""part-d-161497185-H2293-014""","""161497185""","""#insurer-org""",2026-02-01,2026-07-31,"""active""","""4AY0V77DJ35""",2026-06-26 08:38:32.811,"""C:\BCDA_V3\Data\Coverage_v3_04…"


In [32]:
column_names = [
    'coverage_id',
    'active',
    'contained_id',
    'name',
    'resourceType',
    'lastUpdated',
    'filename'
]

df_coverage_contained = (
    df_coverage
    .filter(pl.col('contained').is_not_null())
    .with_columns(
        pl.col('id').alias('coverage_id'),
        pl.col('meta').alias('meta_main')
    )
    .select(
        'coverage_id',
        'contained',
        'meta_main',
        'filename'
    )
    .explode('contained')
    .unnest('contained')
    .with_columns(
        pl.col('id').alias('contained_id'),
        pl.col('meta_main').struct.field('lastUpdated'),
        pl.col('filename').str.split('\\').list.get(-1).alias('filename')
    )
    .select(column_names)
)
df_coverage_contained

coverage_id,active,contained_id,name,resourceType,lastUpdated,filename
str,bool,str,str,str,datetime[μs],str
"""part-a-146341947""",true,"""cms-org""","""Centers for Medicare and Medic…","""Organization""",2026-05-24 07:15:29.760,"""Coverage_v3_0450_2026070619333…"
"""part-b-146341947""",true,"""cms-org""","""Centers for Medicare and Medic…","""Organization""",2026-05-24 07:15:29.760,"""Coverage_v3_0450_2026070619333…"
"""part-d-146341947-S5884-168""",true,"""insurer-org""","""HUMANA PREMIER RX PLAN""","""Organization""",2026-06-26 08:38:32.811,"""Coverage_v3_0450_2026070619333…"
"""dual-146341947""",true,"""cms-org""","""Centers for Medicare and Medic…","""Organization""",2026-05-24 07:15:29.760,"""Coverage_v3_0450_2026070619333…"
"""part-a-175267566""",true,"""cms-org""","""Centers for Medicare and Medic…","""Organization""",2026-06-25 10:12:47.118,"""Coverage_v3_0450_2026070619333…"
…,…,…,…,…,…,…
"""part-b-161497185""",true,"""cms-org""","""Centers for Medicare and Medic…","""Organization""",2026-06-26 08:38:32.769,"""Coverage_v3_0455_2026070619333…"
"""part-c-161497185-H2293-014""",true,"""insurer-org""","""AETNA MEDICARE SIGNATURE EXTRA""","""Organization""",2026-06-26 08:38:32.811,"""Coverage_v3_0455_2026070619333…"
"""part-d-161497185-H2293-014""",true,"""insurer-org""","""AETNA MEDICARE SIGNATURE EXTRA""","""Organization""",2026-06-26 08:38:32.811,"""Coverage_v3_0455_2026070619333…"


In [52]:
df_coverage_class = (
    df_coverage
    .filter(pl.col('class').is_not_null())
    .select(
        'id',
        'class',
        'meta',
        'filename'
    )
    .explode('class')
    .unnest('class')
    .with_columns(
        pl.col('meta').struct.field('lastUpdated'),
        pl.col('type').struct.field('coding').alias('type')
    )
    .explode('type')
    .unnest('type')
    .select(
        'id',
        'code',
        'value',
        'lastUpdated',
        'filename'
    )
)
df_coverage_class

id,code,value,lastUpdated,filename
str,str,str,datetime[μs],str
"""part-a-146341947""","""plan""","""Part A""",2026-05-24 07:15:29.760,"""C:\BCDA_V3\Data\Coverage_v3_04…"
"""part-b-146341947""","""plan""","""Part B""",2026-05-24 07:15:29.760,"""C:\BCDA_V3\Data\Coverage_v3_04…"
"""part-d-146341947-S5884-168""","""plan""","""Part D""",2026-06-26 08:38:32.811,"""C:\BCDA_V3\Data\Coverage_v3_04…"
"""part-d-146341947-S5884-168""","""rxid""","""H72522509""",2026-06-26 08:38:32.811,"""C:\BCDA_V3\Data\Coverage_v3_04…"
"""part-d-146341947-S5884-168""","""rxpcn""","""03200000""",2026-06-26 08:38:32.811,"""C:\BCDA_V3\Data\Coverage_v3_04…"
…,…,…,…,…
"""part-d-161497185-H2293-014""","""rxgroup""","""RXAETD""",2026-06-26 08:38:32.811,"""C:\BCDA_V3\Data\Coverage_v3_04…"
"""part-d-161497185-H2293-014""","""rxpcn""","""MEDDAET""",2026-06-26 08:38:32.811,"""C:\BCDA_V3\Data\Coverage_v3_04…"
"""part-d-161497185-H2293-014""","""rxbin""","""610502""",2026-06-26 08:38:32.811,"""C:\BCDA_V3\Data\Coverage_v3_04…"


In [49]:
df_coverage_extension = (
    df_coverage
    .filter(pl.col('extension').is_not_null())
    .select(
        'id',
        'extension',
        'meta',
        'filename'
    )
    .explode('extension')
    .unnest('extension')
    .with_columns(
        pl.col('meta').struct.field('lastUpdated'),
        pl.col('valueCoding').struct.field('code').alias('code'),
        pl.col('valueCoding').struct.field('system').alias('system')
    )
    .select(
        'id',
        'code',
        'system',
        'lastUpdated',
        'filename'
    )
)
df_coverage_extension

id,code,system,lastUpdated,filename
str,str,str,datetime[μs],str
"""part-a-146341947""","""D""","""https://bluebutton.cms.gov/fhi…",2026-05-24 07:15:29.760,"""C:\BCDA_V3\Data\Coverage_v3_04…"
"""part-a-146341947""","""E""","""https://bluebutton.cms.gov/fhi…",2026-05-24 07:15:29.760,"""C:\BCDA_V3\Data\Coverage_v3_04…"
"""part-a-146341947""","""20""","""https://bluebutton.cms.gov/fhi…",2026-05-24 07:15:29.760,"""C:\BCDA_V3\Data\Coverage_v3_04…"
"""part-a-146341947""","""N""","""https://bluebutton.cms.gov/fhi…",2026-05-24 07:15:29.760,"""C:\BCDA_V3\Data\Coverage_v3_04…"
"""part-a-146341947""","""Y""","""https://bluebutton.cms.gov/fhi…",2026-05-24 07:15:29.760,"""C:\BCDA_V3\Data\Coverage_v3_04…"
…,…,…,…,…
"""part-b-179476241""","""Y""","""https://bluebutton.cms.gov/fhi…",2026-06-27 12:01:02.441,"""C:\BCDA_V3\Data\Coverage_v3_04…"
"""part-b-179476241""","""10""","""https://bluebutton.cms.gov/fhi…",2026-06-27 12:01:02.441,"""C:\BCDA_V3\Data\Coverage_v3_04…"
"""part-b-179476241""","""N""","""https://bluebutton.cms.gov/fhi…",2026-06-27 12:01:02.441,"""C:\BCDA_V3\Data\Coverage_v3_04…"


In [15]:
from datetime import datetime
now = datetime.now().strftime('%d/%m/%Y %H:%M:%S')
def process_eob_base(df_eob: pl.DataFrame):  
    
    def safe_expr(df, source_col, expr, alias):
        if source_col in df.columns:
            return expr.alias(alias)
        return pl.lit(None).alias(alias)
    column_names = [
            'claim_id',
            'cntrl_num',
            'billablePeriod_start',
            'billablePeriod_end',
            'payment_amount',
            'payment_date',
            'patient_id',
            'provider_id',
            'outcome',
            'type_code',
            'type_display',
            'type_code_display',
            'related_value',
            'related_relationship',
            'source',
            'final_action',
            'created',
            'lastUpdated',
            'filename',
            'extract_date'
    ]
    
    df_eob_base = (
        df_eob
        .explode('identifier')
        .with_columns(
            pl.col('identifier').struct.field('value').alias('cntrl_num'),
            pl.col('identifier').struct.field('system').alias('idt_system'),
            pl.col('id').alias('claim_id'),
            pl.col('filename').str.split('\\').list.get(-1).alias('filename')
        )
        .filter(pl.col('idt_system').is_not_null())
        .with_columns(
            pl.lit(now).alias('extract_date')
        )
    )
    
    expres = [
        safe_expr(
            df_eob_base,
            'billablePeriod',
            pl.col('billablePeriod').struct.field('start'),
            'billablePeriod_start'
        ),
        safe_expr(
            df_eob_base,
            'billablePeriod',
            pl.col('billablePeriod').struct.field('end'),
            'billablePeriod_end'
        ),
        safe_expr(
            df_eob_base,
            'payment',
            pl.col('payment').struct.field('amount').struct.field('value'),
            'payment_amount'
        ),
        safe_expr(
            df_eob_base,
            'payment',
            pl.col('payment').struct.field('date'),
            'payment_date'
        ),
        safe_expr(
            df_eob_base,
            'patient',
            pl.col('patient').struct.field('reference').str.split('/').list.get(1),
            'patient_id'
        ),
        safe_expr(
            df_eob_base,
            'provider',
            pl.col('provider').struct.field('reference').str.replace('#',''),
            'provider_id'
        ),
        safe_expr(
            df_eob_base,
            'type',
            pl.col('type').struct.field('coding').list.get(0).struct.field('code'),
            'type_code'
        ),
        safe_expr(
            df_eob_base,
            'type',
            pl.col('type').struct.field('coding').list.get(0).struct.field('display'),
            'type_display'
        ),
        safe_expr(
            df_eob_base,
            'type',
            pl.col('type').struct.field('coding').list.get(1).struct.field('code'),
            'type_code_display'
        ),
        safe_expr(
            df_eob_base,
            'meta',
            pl.col('meta').struct.field('lastUpdated'),
            'lastUpdated'
        ),
        safe_expr(
            df_eob_base,
            'meta',
            pl.col('meta').struct.field('tag').list.get(0).struct.field('code'),
            'source'
        ),
        safe_expr(
            df_eob_base,
            'meta',
            pl.col('meta').struct.field('tag').list.get(1).struct.field('code'),
            'final_action'
        ),
        safe_expr(
            df_eob_base,
            'related',
            pl.col('related').list.get(0).struct.field('reference').struct.field('value'),
            'related_value'
        ),
        safe_expr(
            df_eob_base,
            'related',
            pl.col('related').list.get(0).struct.field('relationship').struct.field('coding').list.get(0).struct.field('code'),
            'related_relationship'
        ),
    ]
    
    df_eob_base = (
        df_eob_base
        .with_columns(expres)
        .select(column_names)
    )
    
    return df_eob_base.to_pandas()

process_eob_base(df_eob)

,claim_id,cntrl_num,billablePeriod_start,billablePeriod_end,payment_amount,payment_date,patient_id,provider_id,outcome,type_code,type_display,type_code_display,related_value,related_relationship,source,final_action,created,lastUpdated,filename,extract_date
0,511819681320,4756869170498930764756869170498930762FGA,2026-06-02,2026-06-02,NaN,NaT,176494789,1205448255,complete,1,MEDICARE PART D ORIGINAL CLAIM,pharmacy,None,None,DDPS,FinalAction,2026-06-04,2026-06-05 03:44:07.098,ExplanationOfBenefit_v3_0025_20260701181421.nd...,06/07/2026 15:36:19
1,511819681483,7262935802498936637262935802498936632FGA,2026-06-02,2026-06-02,NaN,NaT,176494789,1205448255,complete,1,MEDICARE PART D ORIGINAL CLAIM,pharmacy,None,None,DDPS,FinalAction,2026-06-04,2026-06-05 03:44:07.098,ExplanationOfBenefit_v3_0025_20260701181421.nd...,06/07/2026 15:36:19
2,512023760208,4539300282219937434539300282219937432FGA,2026-06-23,2026-06-23,NaN,NaT,176494789,1205448255,complete,1,MEDICARE PART D ORIGINAL CLAIM,pharmacy,None,None,DDPS,FinalAction,2026-06-25,2026-06-26 01:34:31.686,ExplanationOfBenefit_v3_0025_20260701181421.nd...,06/07/2026 15:36:19
3,1006337976064,452926147179190,2026-05-19,2026-05-19,205.97,2026-05-30,176494789,1619977295,complete,71,MEDICARE PART B PROFESSIONAL PHYSICIAN/SUPPLIE...,professional,None,None,NationalClaimsHistory,FinalAction,2026-06-05,2026-06-09 01:21:00.641,ExplanationOfBenefit_v3_0025_20260701181421.nd...,06/07/2026 15:36:19
4,1006337976078,452926147380840,2026-04-07,2026-04-07,74.04,2026-06-03,176494789,1619977295,complete,71,MEDICARE PART B PROFESSIONAL PHYSICIAN/SUPPLIE...,professional,None,None,NationalClaimsHistory,FinalAction,2026-06-05,2026-06-09 01:21:00.641,ExplanationOfBenefit_v3_0025_20260701181421.nd...,06/07/2026 15:36:19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
719,512001788219,3395039311809932233395039311809932232BUA,2026-06-18,2026-06-18,NaN,2026-06-30,173121213,1326426768,complete,1,MEDICARE PART D ORIGINAL CLAIM,pharmacy,None,None,DDPS,NotFinalAction,2026-06-22,2026-06-25 01:53:15.646,ExplanationOfBenefit_v3_0025_20260701181421.nd...,06/07/2026 15:36:19
720,512001791498,6235850523509934466235850523509934462BUA,2026-06-15,2026-06-15,NaN,NaT,173121213,1326426768,complete,3,MEDICARE PART D DELETED CLAIM,pharmacy,None,None,DDPS,NotFinalAction,2026-06-22,2026-06-23 03:17:14.558,ExplanationOfBenefit_v3_0025_20260701181421.nd...,06/07/2026 15:36:19
721,512019068259,3395039311809932233395039311809932232BUA,2026-06-18,2026-06-18,NaN,NaT,173121213,1326426768,complete,3,MEDICARE PART D DELETED CLAIM,pharmacy,None,None,DDPS,NotFinalAction,2026-06-24,2026-06-25 02:08:09.177,ExplanationOfBenefit_v3_0025_20260701181421.nd...,06/07/2026 15:36:19
722,512055721304,8609008762019931148609008762019931142BUA,2026-06-21,2026-06-21,NaN,2026-07-07,173121213,1558443911,complete,1,MEDICARE PART D ORIGINAL CLAIM,pharmacy,None,None,DDPS,FinalAction,2026-06-29,2026-06-30 03:19:16.424,ExplanationOfBenefit_v3_0025_20260701181421.nd...,06/07/2026 15:36:19


In [ ]:
def process_eob_base(df_eob: pl.DataFrame):        
    df_eob_base = (
        df_eob
        .select(
            'id',
            'billablePeriod',
            'created',
            'payment',
            'patient',
            'provider',
            'outcome',
            'identifier',
            'insurance',
            'type',
            'meta',
            'related',
            'filename'
        )
        .explode('identifier')
        .with_columns(
            pl.col('identifier').struct.field('value').alias('cntrl_num'),
            pl.col('identifier').struct.field('system').alias('idt_system'),
        )
        .filter(pl.col('idt_system').is_not_null())
        .with_columns(
            pl.col('billablePeriod').struct.field('start').alias('billablePeriod_start'),
            pl.col('billablePeriod').struct.field('end').alias('billablePeriod_end'),
            pl.col('payment').struct.field('amount').alias('payment_amount'),
            pl.col('payment').struct.field('date').alias('payment_date'),
            pl.col('patient').struct.field('reference').str.split('/').list.get(1).alias('patient_id'),
            pl.col('provider').struct.field('reference').str.replace('#','').alias('provider_id'),
            pl.col('type').struct.field('coding').list.get(0).struct.field('code').alias('type_code'),
            pl.col('type').struct.field('coding').list.get(0).struct.field('display').alias('type_display'),
            pl.col('type').struct.field('coding').list.get(1).struct.field('code').alias('type_code_display'),
            pl.col('meta').struct.field('lastUpdated'),
            pl.col('meta').struct.field('tag').list.get(0).struct.field('code').alias('source'),
            pl.col('meta').struct.field('tag').list.get(1).struct.field('code').alias('final_action'),
            pl.col('related').list.get(0).struct.field('reference'),
            pl.col('related').list.get(0).struct.field('relationship'),
            pl.lit(now).alias('extract_date')
        )
        .with_columns(
            pl.col('payment_amount').struct.field('value').alias('payment_amount'),
            pl.col('reference').struct.field('value').alias('related_value'),
            pl.col('relationship').struct.field('coding').list.get(0).struct.field('code').alias('related_relationship')
        )
        .select(
            'id',
            'cntrl_num',
            'billablePeriod_start',
            'billablePeriod_end',
            'payment_amount',
            'payment_date',
            'patient_id',
            'provider_id',
            'outcome',
            'type_code',
            'type_display',
            'type_code_display',
            'related_value',
            'related_relationship',
            'source',
            'final_action',
            'created',
            'lastUpdated',
            'filename',
            'extract_date'
        )
    )
    return df_eob_base.to_pandas()

id,cntrl_num,billablePeriod_start,billablePeriod_end,payment_amount,payment_date,patient_id,provider_id,outcome,type_code,type_display,type_code_display,related_value,related_relationship,source,final_action,created,lastUpdated,filename
str,str,date,date,f64,date,str,str,str,str,str,str,str,str,str,str,datetime[μs],datetime[μs],str
"""1004324545686""","""452224295902980""",2024-10-14,2024-10-14,0.0,2024-10-24,"""176077134""","""1760452767""","""complete""","""71""","""MEDICARE PART B PROFESSIONAL P…","""professional""",null,null,"""NationalClaimsHistory""","""FinalAction""",2024-10-25 00:00:00,2026-03-08 09:36:25.384,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004357821502""","""453224296753690""",2024-09-05,2024-09-05,0.0,2024-10-25,"""176077134""","""1780654301""","""complete""","""71""","""MEDICARE PART B PROFESSIONAL P…","""professional""",null,null,"""NationalClaimsHistory""","""FinalAction""",2024-11-01 00:00:00,2026-03-08 11:46:00.861,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004357822988""","""22429801807107TXA""",2024-09-05,2024-09-05,0.0,2024-11-07,"""176077134""","""1285798918""","""complete""","""40""","""MEDICARE OUTPATIENT CLAIM""","""institutional""",null,null,"""NationalClaimsHistory""","""FinalAction""",2024-11-01 00:00:00,2026-03-07 10:43:55.187,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004377065539""","""452224305400900""",2024-09-24,2024-09-24,0.0,2024-11-03,"""176077134""","""1306845961""","""complete""","""71""","""MEDICARE PART B PROFESSIONAL P…","""professional""",null,null,"""NationalClaimsHistory""","""FinalAction""",2024-11-08 00:00:00,2026-03-09 01:19:56.899,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004377066394""","""22429700102107NTA""",2024-09-04,2024-09-04,194.87,2024-11-04,"""176077134""","""1528026267""","""complete""","""40""","""MEDICARE OUTPATIENT CLAIM""","""institutional""","""22428303173207NTA""","""prior""","""NationalClaimsHistory""","""FinalAction""",2024-11-08 00:00:00,2026-03-07 18:14:15.038,"""C:\BCDA_V3\Data\ExplanationOfB…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""1006358900507""","""22615400713804ARA""",2026-05-27,2026-05-27,97.59,2026-06-17,"""99908141""","""1477549756""","""complete""","""40""","""MEDICARE OUTPATIENT CLAIM""","""institutional""",null,null,"""NationalClaimsHistory""","""FinalAction""",2026-06-12 00:00:00,2026-06-16 02:10:49.155,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1006408970910""","""520226167312860""",2026-05-26,2026-05-26,12.56,2026-06-19,"""99908141""","""1023838539""","""complete""","""71""","""MEDICARE PART B PROFESSIONAL P…","""professional""",null,null,"""NationalClaimsHistory""","""FinalAction""",2026-06-26 00:00:00,2026-06-30 01:49:56.241,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1006337971749""","""452826149142220""",2026-05-27,2026-05-27,87.8,2026-06-03,"""164660944""","""1164419727""","""complete""","""71""","""MEDICARE PART B PROFESSIONAL P…","""professional""",null,null,"""NationalClaimsHistory""","""FinalAction""",2026-06-05 00:00:00,2026-06-09 01:21:00.072,"""C:\BCDA_V3\Data\ExplanationOfB…"


In [18]:
def process_eob_diagnosis(df_eob: pl.DataFrame):
    df_eob_diagnosis = (
        df_eob
        .filter(pl.col('diagnosis').is_not_null())
        .select(
            'id',
            'diagnosis',
            'meta',
            'patient',
            'filename'
        )
        .pipe(flatten, 'diagnosis')
        .with_columns(
            pl.col('diagnosisCodeableConcept').struct.field('coding').list.get(0).struct.field('code').alias('diagnosis_code'),
            pl.col('diagnosisCodeableConcept').struct.field('coding').list.get(0).struct.field('system').str.split('/').list.get(-1).alias('diagnosis_system'),
            pl.col('type').list.get(0).struct.field('coding').list.get(0).struct.field('code').alias('diagnosis_type_code'),
            pl.col('meta').struct.field('lastUpdated').alias('lastUpdated'),
            pl.col('patient').struct.field('reference').str.split('/').list.get(1).alias('patient_id'),
            pl.col('onAdmission').struct.field('coding').list.get(0).struct.field('code').alias('onAdmission'),
            pl.lit(now).alias('extract_date'),
            pl.col('filename').str.split('\\').list.get(-1).alias('filename')
        )
        .select(
            'id',
            'diagnosis_code',
            'diagnosis_system',
            'diagnosis_type_code',
            'onAdmission',
            'patient_id',
            'lastUpdated',
            'filename',
            'extract_date'
        )
    )
    return df_eob_diagnosis.to_pandas()
process_eob_diagnosis(df_eob)

NameError: name 'flatten' is not defined

In [16]:
from datetime import datetime

now = datetime.now().strftime('%d/%m/%Y %H:%M:%S')

def process_eob_diagnosis(df_eob: pl.DataFrame):
    
    def safe_expr(df, source_col, expr, alias):
        if source_col in df.columns:
            return expr.alias(alias)
        return pl.lit(None).alias(alias)
    
    column_names = [
            'id',
            'diagnosis_code',
            'diagnosis_system',
            'diagnosis_type_code',
            'onAdmission',
            'patient_id',
            'lastUpdated',
            'filename',
            'extract_date'
    ]
    
    df_eob_diagnosis = (
        df_eob
        .filter(pl.col('diagnosis').is_not_null())
        .select(
            'id',
            'diagnosis',
            'meta',
            'patient',
            'filename'
        )
        .pipe(flatten, 'diagnosis')
        .with_columns(
            pl.lit(now).alias('extract_date')
        )
    )
    
    epres = [
        safe_expr(
            df_eob_diagnosis,
            'diagnosisCodeableConcept',
            pl.col('diagnosisCodeableConcept').struct.field('coding').list.get(0).struct.field('code'),
            'diagnosis_code'
        ),
        safe_expr(
            df_eob_diagnosis,
            'diagnosisCodeableConcept',
            pl.col('diagnosisCodeableConcept').struct.field('coding').list.get(0).struct.field('system').str.split('/').list.get(-1),
            'diagnosis_system'
        ),
        safe_expr(
            df_eob_diagnosis,
            'type',
            pl.col('type').list.get(0).struct.field('coding').list.get(0).struct.field('code'),
            'diagnosis_type_code'
        ),
        safe_expr(
            df_eob_diagnosis,
            'meta',
            pl.col('meta').struct.field('lastUpdated'),
            'lastUpdated'
        ),
        safe_expr(
            df_eob_diagnosis,
            'patient',
            pl.col('patient').struct.field('reference').str.split('/').list.get(1),
            'patient_id'
        ),
        safe_expr(
            df_eob_diagnosis,
            'onAdmission',
            pl.col('onAdmission').struct.field('coding').list.get(0).struct.field('code'),
            'onAdmission'
        ),
    ]
    
    df_eob_diagnosis = (
        df_eob_diagnosis
        .with_columns(epres)
        .select(column_names)
    )
    return df_eob_diagnosis.to_pandas()

process_eob_diagnosis(df_eob)

NameError: name 'flatten' is not defined

In [156]:
def process_eob_procedure(df_eob: pl.DataFrame):
    
    def safe_expr(df, source_col, expr, alias):
        if source_col in df.columns:
            return expr.alias(alias)
        return pl.lit(None).alias(alias)
    
    column_names = [
            'id',
            'sequence',
            'procedure_code',
            'code_system',
            'type_code',
            'patient_id',
            'lastUpdated',
            'filename',
            'extract_date'
    ]
    
    df_eob_procedure = (
        df_eob
        .filter(pl.col('procedure').is_not_null())
        .select(
            'id',
            'procedure',
            'meta',
            'patient',
            'filename'
        )
        .pipe(flatten, 'procedure')
        .with_columns(
            pl.lit(now).alias('extract_date')
        )
    )
    
    expres = [
        safe_expr(
            df_eob_procedure,
            'procedureCodeableConcept',
            pl.col('procedureCodeableConcept').struct.field('coding').list.get(0).struct.field('code'),
            'procedure_code'
        ),
        safe_expr(
            df_eob_procedure,
            'procedureCodeableConcept',
            pl.col('procedureCodeableConcept').struct.field('coding').list.get(0).struct.field('system').str.split('/').list.get(-1),
            'code_system'
        ),
        safe_expr(
            df_eob_procedure,
            'type',
            pl.col('type').list.get(0).struct.field('coding').list.get(0).struct.field('code'),
            'type_code'
        ),
        safe_expr(
            df_eob_procedure,
            'meta',
            pl.col('meta').struct.field('lastUpdated'),
            'lastUpdated'
        ),
        safe_expr(
            df_eob_procedure,
            'patient',
            pl.col('patient').struct.field('reference').str.split('/').list.get(1),
            'patient_id'
        )
    ]
    
    df_eob_procedure = (
        df_eob_procedure
        .with_columns(expres)
        .select(column_names)
    )
    
    return df_eob_procedure.to_pandas()

process_eob_procedure(df_eob)

,id,sequence,procedure_code,code_system,type_code,patient_id,lastUpdated,filename,extract_date
0,1002285493712,1,0SP004Z,ICD10,principal,145393328,2026-04-30 18:28:59.785,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0137_2...,06/07/2026 15:49:33
1,1002285493712,2,0SG1071,ICD10,other,145393328,2026-04-30 18:28:59.785,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0137_2...,06/07/2026 15:49:33
2,1002285493712,3,01NB0ZZ,ICD10,other,145393328,2026-04-30 18:28:59.785,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0137_2...,06/07/2026 15:49:33
3,1002285493712,4,07DR3ZZ,ICD10,other,145393328,2026-04-30 18:28:59.785,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0137_2...,06/07/2026 15:49:33
4,1002285493712,5,4A11X4G,ICD10,other,145393328,2026-04-30 18:28:59.785,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0137_2...,06/07/2026 15:49:33
5,1002623273216,1,0SR902Z,ICD10,principal,145393328,2026-04-30 18:28:59.785,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0137_2...,06/07/2026 15:49:33
6,1001453297593,1,0QS906Z,ICD10,principal,108045823,2026-04-30 18:28:59.785,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0137_2...,06/07/2026 15:49:33
7,1002623975383,1,06CN3ZZ,ICD10,principal,147614476,2026-04-30 18:28:59.785,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0137_2...,06/07/2026 15:49:33
8,1002623975383,2,06CY3ZZ,ICD10,other,147614476,2026-04-30 18:28:59.785,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0137_2...,06/07/2026 15:49:33
9,1001429491450,1,01NB0ZZ,ICD10,principal,120266696,2026-04-30 18:28:59.785,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0137_2...,06/07/2026 15:49:33


In [ ]:
def process_eob_procedure(df_eob: pl.DataFrame):
    df_eob_procedure = (
        df_eob
        .filter(pl.col('procedure').is_not_null())
        .select(
            'id',
            'procedure',
            'meta',
            'patient',
            'filename'
        )
        .pipe(flatten, 'procedure')
        .with_columns(
            pl.col('procedureCodeableConcept').struct.field('coding').list.get(0).struct.field('code').alias('procedure_code'),
            pl.col('procedureCodeableConcept').struct.field('coding').list.get(0).struct.field('system').alias('code_system'),
            pl.col('type').list.get(0).struct.field('coding').list.get(0).struct.field('code').alias('type_code'),
            pl.col('meta').struct.field('lastUpdated').alias('lastUpdated'),
            pl.col('patient').struct.field('reference').str.split('/').list.get(1).alias('patient_id'),
            pl.lit(now).alias('extract_date')
        )
        .select(
            'id',
            'procedure_code',
            'code_system',
            'type_code',
            'patient_id',
            'lastUpdated',
            'filename',
            'extract_date'
        )
    )
    return df_eob_procedure.to_pandas()

id,procedure_code,code_system,type_code,patient_id,lastUpdated,filename
str,str,str,str,str,datetime[μs],str
"""93248393639""","""0SG10AJ""","""http://www.cms.gov/Medicare/Co…","""principal""","""109664215""",2026-03-20 04:31:10.845,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""93248393639""","""0SG1071""","""http://www.cms.gov/Medicare/Co…","""other""","""109664215""",2026-03-20 04:31:10.845,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""93248393639""","""0SG30AJ""","""http://www.cms.gov/Medicare/Co…","""other""","""109664215""",2026-03-20 04:31:10.845,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""93248393639""","""0QB33ZZ""","""http://www.cms.gov/Medicare/Co…","""other""","""109664215""",2026-03-20 04:31:10.845,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""93248393639""","""30233N1""","""http://www.cms.gov/Medicare/Co…","""other""","""109664215""",2026-03-20 04:31:10.845,"""C:\BCDA_V3\Data\ExplanationOfB…"
…,…,…,…,…,…,…
"""1006384979857""","""0D7G8ZZ""","""http://www.cms.gov/Medicare/Co…","""principal""","""152421027""",2026-06-23 01:55:18.077,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1006384979857""","""0D7F8ZZ""","""http://www.cms.gov/Medicare/Co…","""other""","""152421027""",2026-06-23 01:55:18.077,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1006384979857""","""0DBG8ZX""","""http://www.cms.gov/Medicare/Co…","""other""","""152421027""",2026-06-23 01:55:18.077,"""C:\BCDA_V3\Data\ExplanationOfB…"


In [ ]:
def process_eob_total(df_eob: pl.DataFrame):
    
    def safe_expr(df, source_col, expr, alias):
        if source_col in df.columns:
            return expr.alias(alias)
        return pl.lit(None).alias(alias)
    
    column_names = [
            'id',
            'total_amount',
            'category_code',
            'patient_id',
            'lastUpdated',
            'filename',
            'extract_date'
    ]
    
    df_eob_total = (
        df_eob
        .filter(pl.col('total').is_not_null())
        .select(
            'id',
            'total',
            'patient',
            'meta',
            'filename'
        )
        .pipe(flatten, 'total')
        .with_columns(
            pl.lit(now).alias('extract_date')
        )
    )
    
    expres = [
        safe_expr(
            df_eob_total,
            'amount',
            pl.col('amount').struct.field('value'),
            'total_amount'
        ),
        safe_expr(
            df_eob_total,
            'patient',
            pl.col('patient').struct.field('reference').str.split('/').list.get(1),
            'patient_id'
        ),
        safe_expr(
            df_eob_total,
            'meta',
            pl.col('meta').struct.field('lastUpdated'),
            'lastUpdated'
        ),
        safe_expr(
            df_eob_total,
            'category',
            pl.col('category').struct.field('coding').list.get(1).struct.field('display'),
            'category_code'
        ),
    ]
    
    df_eob_total = (
        df_eob_total
        .with_columns(expres)
        .select(column_names)
    )
    
    return df_eob_total.to_pandas()

process_eob_total(df_eob)

,id,total_amount,category_code,patient_id,lastUpdated,filename,extract_date
0,1004324545686,109.40,CLM_ALOWD_CHRG_AMT,176077134,2026-03-08 09:36:25.384,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 08:17:19
1,1004324545686,359.00,CLM_SBMT_CHRG_AMT,176077134,2026-03-08 09:36:25.384,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 08:17:19
2,1004324545686,0.00,CLM_BENE_PMT_AMT,176077134,2026-03-08 09:36:25.384,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 08:17:19
3,1004324545686,0.00,CLM_PRVDR_PMT_AMT,176077134,2026-03-08 09:36:25.384,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 08:17:19
4,1004324545686,109.40,CLM_MDCR_DDCTBL_AMT,176077134,2026-03-08 09:36:25.384,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 08:17:19
...,...,...,...,...,...,...,...
971382,1006359631211,66.99,CLM_ALOWD_CHRG_AMT,164660944,2026-06-16 02:21:57.152,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0449_2...,06/07/2026 08:17:19
971383,1006359631211,488.00,CLM_SBMT_CHRG_AMT,164660944,2026-06-16 02:21:57.152,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0449_2...,06/07/2026 08:17:19
971384,1006359631211,0.00,CLM_BENE_PMT_AMT,164660944,2026-06-16 02:21:57.152,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0449_2...,06/07/2026 08:17:19
971385,1006359631211,52.95,CLM_PRVDR_PMT_AMT,164660944,2026-06-16 02:21:57.152,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0449_2...,06/07/2026 08:17:19


In [ ]:
def process_eob_total(df_eob: pl.DataFrame):
    df_eob_total = (
        df_eob
        .filter(pl.col('total').is_not_null())
        .select(
            'id',
            'total',
            'patient',
            'meta',
            'filename'
        )
        .pipe(flatten, 'total')
        .with_columns(
            pl.col('amount').struct.field('value').alias('total_amount'),
            pl.col('patient').struct.field('reference').str.split('/').list.get(1).alias('patient_id'),
            pl.col('meta').struct.field('lastUpdated').alias('lastUpdated'),
            pl.col('category').struct.field('coding').list.get(1).struct.field('code').alias('category_code'),
            pl.lit(now).alias('extract_date')
        )
        .select(
            'id',
            'total_amount',
            'category_code',
            'patient_id',
            'lastUpdated',
            'filename',
            'extract_date'
        )
    )
    return df_eob_total.to_pandas()

id,total_amount,category_code,patient_id,lastUpdated,filename
str,f64,str,str,datetime[μs],str
"""1004324545686""",109.4,"""CLM_ALOWD_CHRG_AMT""","""176077134""",2026-03-08 09:36:25.384,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004324545686""",359.0,"""CLM_SBMT_CHRG_AMT""","""176077134""",2026-03-08 09:36:25.384,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004324545686""",0.0,"""CLM_BENE_PMT_AMT""","""176077134""",2026-03-08 09:36:25.384,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004324545686""",0.0,"""CLM_PRVDR_PMT_AMT""","""176077134""",2026-03-08 09:36:25.384,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004324545686""",109.4,"""CLM_MDCR_DDCTBL_AMT""","""176077134""",2026-03-08 09:36:25.384,"""C:\BCDA_V3\Data\ExplanationOfB…"
…,…,…,…,…,…
"""1006359631211""",66.99,"""CLM_ALOWD_CHRG_AMT""","""164660944""",2026-06-16 02:21:57.152,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1006359631211""",488.0,"""CLM_SBMT_CHRG_AMT""","""164660944""",2026-06-16 02:21:57.152,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1006359631211""",0.0,"""CLM_BENE_PMT_AMT""","""164660944""",2026-06-16 02:21:57.152,"""C:\BCDA_V3\Data\ExplanationOfB…"


In [38]:
from datetime import datetime
MIN_SQL_DATE = datetime(1753, 1, 1)

df_eob_item = (
    df_eob
    .filter(pl.col('item').is_not_null())
    .select(
        'id',
        'item',
        'patient',
        'meta',
        'filename'
    )
    .explode('item')
    .unnest('item')
    .with_columns(
        pl.col('productOrService').struct.field('coding').list.get(0).struct.field('code').alias('product_service_code'),
        pl.col('productOrService').struct.field('coding').list.get(0).struct.field('system').str.split('/').list.get(-1).alias('product_service_system'),
        pl.col('quantity').struct.field('value').alias('quantity_value'),
        pl.col('patient').struct.field('reference').str.split('/').list.get(1).alias('patient_id'),
        pl.col('meta').struct.field('lastUpdated').alias('lastUpdated'),
        pl.col('revenue').struct.field('coding').list.get(0).struct.field('code').alias('revenue_code'),
        pl.col('revenue').struct.field('coding').list.get(0).struct.field('display').alias('revenue_display'),
        pl.col('servicedPeriod').struct.field('start').alias('service_start'),
        pl.col('servicedPeriod').struct.field('end').alias('service_end'),
        pl.col('locationCodeableConcept').struct.field('coding').list.get(0).struct.field('code').alias('location_code'),
        pl.col('locationCodeableConcept').struct.field('coding').list.get(0).struct.field('display').alias('location_display'),
        pl.col('diagnosisSequence').list.get(0).alias('diagnosisSequence'),
        pl.col('informationSequence').list.get(0).alias('informationSequence'),
        pl.col('modifier').list.get(0).struct.field('coding').list.get(0).struct.field('code').alias('modifier'),
        pl.when(pl.col('servicedDate') < MIN_SQL_DATE)
        .then(None)
        .otherwise(pl.col('servicedDate'))
        .alias('servicedDate')
    )
    .select(
        'id',
        'sequence',
        'diagnosisSequence',
        'informationSequence',
        'location_code',
        'location_display',
        'product_service_code',
        'modifier',
        'product_service_system',
        'quantity_value',
        'revenue_code',
        'revenue_display',
        'service_start',
        'service_end',
        'servicedDate',
        'patient_id',
        'lastUpdated',
        'filename'
    )
)
df_eob_item

id,sequence,diagnosisSequence,informationSequence,location_code,location_display,product_service_code,modifier,product_service_system,quantity_value,revenue_code,revenue_display,service_start,service_end,servicedDate,patient_id,lastUpdated,filename
str,i64,i64,i64,str,str,str,str,str,f64,str,str,date,date,date,str,datetime[μs],str
"""1004324545686""",1,1,3,"""10""","""Telehealth Provided in Patient…","""99214""",null,"""cpt""",1.0,null,null,2024-10-14,2024-10-14,null,"""176077134""",2026-03-08 09:36:25.384,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004357821502""",1,1,3,"""10""","""Telehealth Provided in Patient…","""99213""","""95""","""cpt""",1.0,null,null,2024-09-05,2024-09-05,null,"""176077134""",2026-03-08 11:46:00.861,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004357822988""",1,null,null,null,null,"""Q3014""",null,"""HCPCSReleaseCodeSets""",1.0,"""0780""","""TELEMEDICINE - GENERAL CLASSIF…",2024-09-05,2024-09-05,null,"""176077134""",2026-03-07 10:43:55.187,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004357822988""",2,null,null,null,null,"""not-applicable""",null,"""data-absent-reason""",0.0,"""0001""","""TOTAL CHARGE""",null,null,null,"""176077134""",2026-03-07 10:43:55.187,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004377065539""",1,1,3,"""15""","""Mobile Unit""","""93000""",null,"""cpt""",1.0,null,null,2024-09-24,2024-09-24,null,"""176077134""",2026-03-09 01:19:56.899,"""C:\BCDA_V3\Data\ExplanationOfB…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""1006408970910""",2,2,4,"""11""","""Office""","""81002""","""59""","""cpt""",1.0,null,null,2026-05-26,2026-05-26,null,"""99908141""",2026-06-30 01:49:56.241,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1006337971749""",1,1,3,"""11""","""Office""","""99214""",null,"""cpt""",1.0,null,null,2026-05-27,2026-05-27,null,"""164660944""",2026-06-09 01:21:00.072,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1006337972436""",1,null,null,null,null,"""72141""","""TC""","""cpt""",1.0,"""0610""","""MAGNETIC RESONANCE TECHNOLOGY …",2026-05-27,2026-05-27,null,"""164660944""",2026-06-09 01:01:43.513,"""C:\BCDA_V3\Data\ExplanationOfB…"


In [ ]:
from datetime import datetime

now = datetime.now().strftime('%d/%m/%Y %H:%M:%S')

def process_eob_item(df_eob: pl.DataFrame):
    
    
    def safe_expr(df, source_col, expr, alias):
        if source_col in df.columns:
            return expr.alias(alias)
        return pl.lit(None).alias(alias)
    

    column_names = [
            'id',
            'sequence',
            'diagnosisSequence',
            'informationSequence',
            'location_code',
            'location_display',
            'product_service_code',
            'modifier',
            'product_service_system',
            'quantity_value',
            'revenue_code',
            'revenue_display',
            'service_start',
            'service_end',
            'servicedDate',
            'patient_id',
            'lastUpdated',
            'filename',
            'extract_date'
    ]
    

    
    df_eob_item = (
        df_eob
        .filter(pl.col('item').is_not_null())
        .select(
            'id',
            'item',
            'patient',
            'meta',
            'filename'
        )
        .pipe(flatten, 'item')
        .with_columns(
            pl.lit(now).alias('extract_date')
        )
    )
    
        
    expr = [
        safe_expr(
            df_eob_item,
            'productOrService',
            pl.col('productOrService').struct.field('coding').list.get(0).struct.field('code'),
            'product_service_code'
        ),
        safe_expr(
            df_eob_item,
            'productOrService',
            pl.col('productOrService').struct.field('coding').list.get(0).struct.field('system').str.split('/').list.get(-1),
            'product_service_system'
        ),   
        safe_expr(
            df_eob_item,
            'quantity',
            pl.col('quantity').struct.field('value'),
            'quantity_value'
        ),
        safe_expr(
            df_eob_item,
            'patient',
            pl.col('patient').struct.field('reference').str.split('/').list.get(1),
            'patient_id'
        ),
        safe_expr(
            df_eob_item,
            'meta',
            pl.col('meta').struct.field('lastUpdated'),
            'lastUpdated'
        ),
        safe_expr(
            df_eob_item,
            'revenue',
            pl.col('revenue').struct.field('coding').list.get(0).struct.field('code'),
            'revenue_code'
        ),
        safe_expr(
            df_eob_item,
            'revenue',
            pl.col('revenue').struct.field('coding').list.get(0).struct.field('display'),
            'revenue_display'
        ),   
        safe_expr(
            df_eob_item,
            'servicedPeriod',
            pl.col('servicedPeriod').struct.field('start'),
            'service_start'
        ),
        safe_expr(
            df_eob_item,
            'servicedPeriod',
            pl.col('servicedPeriod').struct.field('end'),
            'service_end'
        ),
        safe_expr(
            df_eob_item,
            'locationCodeableConcept',
            pl.col('locationCodeableConcept').struct.field('coding').list.get(0).struct.field('code'),
            'location_code'
        ),
        safe_expr(
            df_eob_item,
            'locationCodeableConcept',
            pl.col('locationCodeableConcept').struct.field('coding').list.get(0).struct.field('display'),
            'location_display'
        ),
        safe_expr(
            df_eob_item,
            'diagnosisSequence',
            pl.col('diagnosisSequence').list.get(0),
            'diagnosisSequence'
        ),   
        safe_expr(
            df_eob_item,
            'informationSequence',
            pl.col('informationSequence').list.get(0),
            'informationSequence'
        ),
        safe_expr(
            df_eob_item,
            'modifier',
            pl.col('modifier').list.get(0).struct.field('coding').list.get(0).struct.field('code'),
            'modifier'
        ),
        safe_expr(
            df_eob_item,
            'servicedDate',
            pl.when(pl.col('servicedDate') < MIN_SQL_DATE).then(None).otherwise(pl.col('servicedDate')),
            'servicedDate'
        )
    ]
    df_eob_item = (
        df_eob_item
        .with_columns(expr)
        .select(column_names)
    )
        
    return df_eob_item.to_pandas()



process_eob_item(df_eob)



,id,sequence,diagnosisSequence,informationSequence,location_code,location_display,product_service_code,modifier,product_service_system,quantity_value,revenue_code,revenue_display,service_start,service_end,servicedDate,patient_id,lastUpdated,filename,extract_date
0,1004324545686,1,1.0,3.0,10,Telehealth Provided in Patient’s Home,99214,None,cpt,1.0,None,None,2024-10-14,2024-10-14,NaT,176077134,2026-03-08 09:36:25.384,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 07:59:47
1,1004357821502,1,1.0,3.0,10,Telehealth Provided in Patient’s Home,99213,95,cpt,1.0,None,None,2024-09-05,2024-09-05,NaT,176077134,2026-03-08 11:46:00.861,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 07:59:47
2,1004357822988,1,NaN,NaN,None,None,Q3014,None,HCPCSReleaseCodeSets,1.0,0780,TELEMEDICINE - GENERAL CLASSIFICATION,2024-09-05,2024-09-05,NaT,176077134,2026-03-07 10:43:55.187,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 07:59:47
3,1004357822988,2,NaN,NaN,None,None,not-applicable,None,data-absent-reason,0.0,0001,TOTAL CHARGE,NaT,NaT,NaT,176077134,2026-03-07 10:43:55.187,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 07:59:47
4,1004377065539,1,1.0,3.0,15,Mobile Unit,93000,None,cpt,1.0,None,None,2024-09-24,2024-09-24,NaT,176077134,2026-03-09 01:19:56.899,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 07:59:47
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
488212,1006408970910,2,2.0,4.0,11,Office,81002,59,cpt,1.0,None,None,2026-05-26,2026-05-26,NaT,99908141,2026-06-30 01:49:56.241,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0449_2...,06/07/2026 07:59:47
488213,1006337971749,1,1.0,3.0,11,Office,99214,None,cpt,1.0,None,None,2026-05-27,2026-05-27,NaT,164660944,2026-06-09 01:21:00.072,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0449_2...,06/07/2026 07:59:47
488214,1006337972436,1,NaN,NaN,None,None,72141,TC,cpt,1.0,0610,MAGNETIC RESONANCE TECHNOLOGY (MRT) - GENERAL ...,2026-05-27,2026-05-27,NaT,164660944,2026-06-09 01:01:43.513,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0449_2...,06/07/2026 07:59:47
488215,1006337972436,2,NaN,NaN,None,None,not-applicable,None,data-absent-reason,0.0,0001,TOTAL CHARGE,NaT,NaT,NaT,164660944,2026-06-09 01:01:43.513,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0449_2...,06/07/2026 07:59:47


In [ ]:
def process_eob_adjudication(df_eob: pl.DataFrame):
    def safe_expr(df, source_col, expr, alias):
        if source_col in df.columns:
            return expr.alias(alias)
        return pl.lit(None).alias(alias)
    
    

In [94]:
def process_eob_adjudication(df_eob: pl.DataFrame):
    
    def safe_expr(df, source_col, expr, alias):
        if source_col in df.columns:
            return expr.alias(alias)
        return pl.lit(None).alias(alias)
    
    column_names = [
        'claim_id',
        'category_display',
        'reason_display',
        'amount',
        'value',
        'lastUpdated',
        'filename'
    ]

    df_eob_adjudication = (
        df_eob
        .filter(pl.col('adjudication').is_not_null())
        .select(
            'id',
            'adjudication',
            'meta',
            'filename'
        )
        .explode('adjudication')
        .unnest('adjudication')
        .with_columns(
            pl.col('id').alias('claim_id'),
            pl.col('meta').struct.field('lastUpdated'),
            pl.col('filename').str.split('\\').list.get(-1).alias('filename')
        )
    )
    expres = [
        safe_expr(
            df_eob_adjudication,
            'category',
            pl.col('category').struct.field('coding').list.get(0).struct.field('display'),
            'category_display'
        ),
        safe_expr(
            df_eob_adjudication,
            'reason',
            pl.col('reason').struct.field('coding').list.get(0).struct.field('display'),
            'reason_display'
        ),
        safe_expr(
            df_eob_adjudication,
            'amount',
            pl.col('amount').struct.field('value'),
            'amount'
        )
    ]
    df_eob_adjudication = (
        df_eob_adjudication
        .with_columns(expres)
        .select(column_names)
    )
    return df_eob_adjudication#.to_pandas()

adjudication = process_eob_adjudication(df_eob)
adjudication.filter((pl.col('value').is_not_null()) & (pl.col('value')> 0))

claim_id,category_display,reason_display,amount,value,lastUpdated,filename
str,str,str,f64,f64,datetime[μs],str
"""1002007650463""","""PPS DRG Weight Number""",null,null,3.36,2026-04-30 18:28:59.785,"""ExplanationOfBenefit_v3_0042_2…"
"""1002007650463""","""Claim Medicare Utilization Day…",null,null,1.0,2026-04-30 18:28:59.785,"""ExplanationOfBenefit_v3_0042_2…"
"""1006310511766""","""PPS DRG Weight Number""",null,null,1.94,2026-06-02 07:00:41.781,"""ExplanationOfBenefit_v3_0042_2…"
"""1006310511766""","""Claim Medicare Utilization Day…",null,null,1.0,2026-06-02 07:00:41.781,"""ExplanationOfBenefit_v3_0042_2…"
"""1002435540236""","""PPS DRG Weight Number""",null,null,2.35,2026-04-30 18:28:59.785,"""ExplanationOfBenefit_v3_0042_2…"
…,…,…,…,…,…,…
"""1003034087566""","""Claim Medicare Utilization Day…",null,null,7.0,2026-04-30 18:28:59.785,"""ExplanationOfBenefit_v3_0042_2…"
"""1006385631479""","""PPS DRG Weight Number""",null,null,1.94,2026-06-23 01:55:19.886,"""ExplanationOfBenefit_v3_0042_2…"
"""1006385631479""","""Claim Medicare Utilization Day…",null,null,1.0,2026-06-23 01:55:19.886,"""ExplanationOfBenefit_v3_0042_2…"


In [ ]:
def process_eob_item_adjudication(df_eob: pl.DataFrame):
    
    def safe_expr(df, source_col, expr, alias):
        if source_col in df.columns:
            return expr.alias(alias)
        return pl.lit(None).alias(alias)
    
    column_names = [
            'id',
            'sequence',
            'adjudication_amount',
            'adjudication_category_display',
            'adjudication_reason_display',
            'patient_id',
            'lastUpdated',
            'filename',
            'extract_date'
    ]
    
    df_eob_item_adjudication = (
        df_eob
        .filter(pl.col('item').is_not_null())
        .select(
            'id',
            'item',
            'patient',
            'meta',
            'filename'
        )
        .pipe(flatten, 'item')
        .select(
            'id',
            'sequence',
            'adjudication',
            'patient',
            'meta',
            'filename'
        )
        .filter(pl.col('adjudication').is_not_null())
        .pipe(flatten, 'adjudication')
        .with_columns(
            pl.lit(now).alias('extract_date')
        )
    )
    
    expres = [
        safe_expr(
            df_eob_item_adjudication,
            'amount',
            pl.col('amount').struct.field('value'),
            'adjudication_amount'
        ),
        safe_expr(
            df_eob_item_adjudication,
            'patient',
            pl.col('patient').struct.field('reference').str.split('/').list.get(1),
            'patient_id'
        ),
        safe_expr(
            df_eob_item_adjudication,
            'meta',
            pl.col('meta').struct.field('lastUpdated'),
            'lastUpdated'
        ),
        safe_expr(
            df_eob_item_adjudication,
            'category',
            pl.col('category').struct.field('coding').list.get(0).struct.field('display'),
            'adjudication_category_display'
        ),
        safe_expr(
            df_eob_item_adjudication,
            'reason',
            pl.col('reason').struct.field('coding').list.get(0).struct.field('display'),
            'adjudication_reason_display'
        ),
    ]
    
    df_eob_item_adjudication = (
        df_eob_item_adjudication
        .with_columns(expres)
        .select(column_names)
    )
    
    return df_eob_item_adjudication

adjudication = process_eob_item_adjudication(df_eob)

ColumnNotFoundError: unable to find column "value"; valid columns: ["id", "sequence", "amount", "category", "reason", "patient", "meta", "filename", "extract_date", "adjudication_amount", "patient_id", "lastUpdated", "adjudication_category_display", "adjudication_reason_display"]

In [ ]:
adjudication

In [60]:
adjudication.filter(pl.col('id') == '1001369071601').show(limit=None)

id,sequence,adjudication_amount,adjudication_category_display,adjudication_reason_display,patient_id,lastUpdated,filename,extract_date
str,i64,f64,str,str,str,datetime[μs],str,str
"""1001369071601""",1,0.0,"""Deductible""",null,"""163781595""",2026-04-30 18:28:59.785,"""C:\BCDA_V3\Data\ExplanationOfB…","""06/07/2026 15:49:33"""
"""1001369071601""",1,0.0,"""Blood Deductible Amount""",null,"""163781595""",2026-04-30 18:28:59.785,"""C:\BCDA_V3\Data\ExplanationOfB…","""06/07/2026 15:49:33"""
"""1001369071601""",1,0.0,"""Paid to patient""",null,"""163781595""",2026-04-30 18:28:59.785,"""C:\BCDA_V3\Data\ExplanationOfB…","""06/07/2026 15:49:33"""
"""1001369071601""",1,0.0,"""Paid by patient""",null,"""163781595""",2026-04-30 18:28:59.785,"""C:\BCDA_V3\Data\ExplanationOfB…","""06/07/2026 15:49:33"""
"""1001369071601""",1,0.0,"""Noncovered""",null,"""163781595""",2026-04-30 18:28:59.785,"""C:\BCDA_V3\Data\ExplanationOfB…","""06/07/2026 15:49:33"""
"""1001369071601""",1,0.0,"""Paid to provider""",null,"""163781595""",2026-04-30 18:28:59.785,"""C:\BCDA_V3\Data\ExplanationOfB…","""06/07/2026 15:49:33"""
"""1001369071601""",1,0.0,"""Benefit Amount""",null,"""163781595""",2026-04-30 18:28:59.785,"""C:\BCDA_V3\Data\ExplanationOfB…","""06/07/2026 15:49:33"""
"""1001369071601""",1,5828.0,"""Submitted Amount""",null,"""163781595""",2026-04-30 18:28:59.785,"""C:\BCDA_V3\Data\ExplanationOfB…","""06/07/2026 15:49:33"""
"""1001369071601""",1,0.0,"""Revenue Center Coinsurance/Wag…",null,"""163781595""",2026-04-30 18:28:59.785,"""C:\BCDA_V3\Data\ExplanationOfB…","""06/07/2026 15:49:33"""


In [ ]:
def process_eob_item_adjudication(df_eob: pl.DataFrame):
    
    df_eob_item_adjudication = (
        df_eob
        .filter(pl.col('item').is_not_null())
        .select(
            'id',
            'item',
            'patient',
            'meta',
            'filename'
        )
        .pipe(flatten, 'item')
        .select(
            'id',
            'sequence',
            'adjudication',
            'patient',
            'meta',
            'filename'
        )
        .filter(pl.col('adjudication').is_not_null())
        .pipe(flatten, 'adjudication')
        .with_columns(
            pl.col('amount').struct.field('value').alias('adjudication_amount'),
            pl.col('patient').struct.field('reference').str.split('/').list.get(1).alias('patient_id'),
            pl.col('meta').struct.field('lastUpdated').alias('lastUpdated'),
            pl.col('category').struct.field('coding').list.get(0).struct.field('display').alias('adjudication_category_display'),
            pl.col('reason').struct.field('coding').list.get(0).struct.field('display').alias('adjudication_reason_display'),
            pl.lit(now).alias('extract_date')
        )
        .select(
            'id',
            'sequence',
            'adjudication_amount',
            'adjudication_category_display',
            'adjudication_reason_display',
            'patient_id',
            'lastUpdated',
            'filename',
            'extract_date'
        )
    )
    return df_eob_item_adjudication.to_pandas()

claim_id,category_display,reason_display,amount,lastUpdated,filename
str,str,str,f64,datetime[μs],str
"""511135542926""","""Benefit Payment Status""","""In Network""",null,2026-03-31 18:39:05.937,"""ExplanationOfBenefit_v3_0032_2…"
"""511135559726""","""Benefit Payment Status""","""In Network""",null,2026-04-13 19:10:23.399,"""ExplanationOfBenefit_v3_0032_2…"
"""511135559727""","""Benefit Payment Status""","""In Network""",null,2026-04-13 19:10:23.399,"""ExplanationOfBenefit_v3_0032_2…"
"""511251162334""","""Benefit Payment Status""","""In Network""",null,2026-04-13 19:10:23.399,"""ExplanationOfBenefit_v3_0032_2…"
"""511251162335""","""Benefit Payment Status""","""In Network""",null,2026-04-13 19:10:23.399,"""ExplanationOfBenefit_v3_0032_2…"
…,…,…,…,…,…
"""1006359630822""","""Blood Noncovered Charge Amount""",null,0.0,2026-06-16 02:11:11.310,"""ExplanationOfBenefit_v3_0032_2…"
"""1006385623798""","""Benefit Payment Status""","""Other""",null,2026-06-23 02:26:25.804,"""ExplanationOfBenefit_v3_0032_2…"
"""1006385623798""","""Primary Payer Paid Amount""",null,0.0,2026-06-23 02:26:25.804,"""ExplanationOfBenefit_v3_0032_2…"


In [104]:
def process_eob_item_ext(df_eob: pl.DataFrame):
    
    def safe_expr(df, source_col, expr, alias):
        if source_col in df.columns:
            return expr.alias(alias)
        return pl.lit(None).alias(alias)
    
    column_names = [
            'id',
            'sequence',
            'identifierValue',
            'valueDecimal',
            'extension_code',
            'extension_display',
            'url',
            'patient_id',
            'lastUpdated',
            'filename',
            'extract_date'
    ]
    
    df_eob_item_ext = (
        df_eob
        .filter(pl.col('item').is_not_null())
        .select(
            'id',
            'item',
            'patient',
            'meta',
            'filename'
        )
        .pipe(flatten, 'item')
        .select(
            'id',
            'sequence',
            'extension',
            'patient',
            'meta',
            'filename'
        )
        .filter(pl.col('extension').is_not_null())
        .pipe(flatten, 'extension')
        .with_columns(
            pl.lit(now).alias('extract_date')
        )
    )
    
    expres = [
        safe_expr(
            df_eob_item_ext,
            'valueCoding',
            pl.col('valueCoding').struct.field('code'),
            'extension_code'
        ),
        safe_expr(
            df_eob_item_ext,
            'valueCoding',
            pl.col('valueCoding').struct.field('display'),
            'extension_display'
        ),
        safe_expr(
            df_eob_item_ext,
            'valueIdentifier',
            pl.col('valueIdentifier').struct.field('value'),
            'identifierValue'
        ),
        safe_expr(
            df_eob_item_ext,
            'patient',
            pl.col('patient').struct.field('reference').str.split('/').list.get(1),
            'patient_id'
        ),
        safe_expr(
            df_eob_item_ext,
            'meta',
            pl.col('meta').struct.field('lastUpdated'),
            'lastUpdated'
        )
    ]
    
    df_eob_item_ext = (
        df_eob_item_ext
        .with_columns(expres)
        .select(column_names)
    )
    
    return df_eob_item_ext

df_eob_item_ext = process_eob_item_ext(df_eob)
df_eob_item_ext.filter(pl.col('valueDecimal').is_not_null())

id,sequence,identifierValue,valueDecimal,extension_code,extension_display,url,patient_id,lastUpdated,filename,extract_date
str,i64,str,f64,str,str,str,str,datetime[μs],str,str
"""1006075942385""",1,null,0.0,null,null,"""https://bluebutton.cms.gov/fhi…","""146074456""",2026-04-30 18:29:00.177,"""C:\BCDA_V3\Data\ExplanationOfB…","""06/07/2026 15:49:33"""
"""1006075942385""",1,null,1.0,null,null,"""https://bluebutton.cms.gov/fhi…","""146074456""",2026-04-30 18:29:00.177,"""C:\BCDA_V3\Data\ExplanationOfB…","""06/07/2026 15:49:33"""
"""1005992080409""",1,null,0.0,null,null,"""https://bluebutton.cms.gov/fhi…","""69733319""",2026-03-30 20:40:05.412,"""C:\BCDA_V3\Data\ExplanationOfB…","""06/07/2026 15:49:33"""
"""1005992080409""",1,null,1.0,null,null,"""https://bluebutton.cms.gov/fhi…","""69733319""",2026-03-30 20:40:05.412,"""C:\BCDA_V3\Data\ExplanationOfB…","""06/07/2026 15:49:33"""
"""1005992080409""",2,null,0.0,null,null,"""https://bluebutton.cms.gov/fhi…","""69733319""",2026-03-30 20:40:05.412,"""C:\BCDA_V3\Data\ExplanationOfB…","""06/07/2026 15:49:33"""
…,…,…,…,…,…,…,…,…,…,…
"""1006359611447""",2,null,28.0,null,null,"""https://bluebutton.cms.gov/fhi…","""138993899""",2026-06-16 02:21:56.466,"""C:\BCDA_V3\Data\ExplanationOfB…","""06/07/2026 15:49:33"""
"""1006385604909""",1,null,0.0,null,null,"""https://bluebutton.cms.gov/fhi…","""138993899""",2026-06-23 02:26:15.607,"""C:\BCDA_V3\Data\ExplanationOfB…","""06/07/2026 15:49:33"""
"""1006385604909""",1,null,1.0,null,null,"""https://bluebutton.cms.gov/fhi…","""138993899""",2026-06-23 02:26:15.607,"""C:\BCDA_V3\Data\ExplanationOfB…","""06/07/2026 15:49:33"""


In [ ]:
def process_eob_item_ext(df_eob: pl.DataFrame):
    df_eob_item_ext = (
        df_eob
        .filter(pl.col('item').is_not_null())
        .select(
            'id',
            'item',
            'patient',
            'meta',
            'filename'
        )
        .pipe(flatten, 'item')
        .select(
            'id',
            'sequence',
            'extension',
            'patient',
            'meta',
            'filename'
        )
        .filter(pl.col('extension').is_not_null())
        .pipe(flatten, 'extension')
        .with_columns(
            pl.col('valueCoding').struct.field('code').alias('extension_code'),
            pl.col('valueCoding').struct.field('display').alias('extension_display'),
            pl.col('patient').struct.field('reference').str.split('/').list.get(1).alias('patient_id'),
            pl.col('meta').struct.field('lastUpdated').alias('lastUpdated'),
            pl.lit(now).alias('extract_date')
        )
        .select(
            'id',
            'sequence',
            'extension_code',
            'extension_display',
            'url',
            'patient_id',
            'lastUpdated',
            'filename',
            'extract_date'
        )
    )
    return df_eob_item_ext.to_pandas()

id,sequence,extension_code,extension_display,url,patient_id,lastUpdated,filename
str,i64,str,str,str,str,datetime[μs],str
"""1004324545686""",1,"""1""","""Medical care""","""https://bluebutton.cms.gov/fhi…","""176077134""",2026-03-08 09:36:25.384,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004324545686""",1,"""0""","""80%""","""https://bluebutton.cms.gov/fhi…","""176077134""",2026-03-08 09:36:25.384,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004324545686""",1,"""50""","""NURSE PRACTITIONER""","""https://bluebutton.cms.gov/fhi…","""176077134""",2026-03-08 09:36:25.384,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004324545686""",1,"""0""","""Service Subject to Deductible""","""https://bluebutton.cms.gov/fhi…","""176077134""",2026-03-08 09:36:25.384,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004324545686""",1,null,null,"""https://bluebutton.cms.gov/fhi…","""176077134""",2026-03-08 09:36:25.384,"""C:\BCDA_V3\Data\ExplanationOfB…"
…,…,…,…,…,…,…,…
"""1006359631211""",1,"""0""","""N/A""","""https://bluebutton.cms.gov/fhi…","""164660944""",2026-06-16 02:21:57.152,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1006359631211""",1,"""A""","""Allowed""","""https://bluebutton.cms.gov/fhi…","""164660944""",2026-06-16 02:21:57.152,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1006359631211""",1,"""1""","""PHYSICIANS OR SUPPLIERS BILLIN…","""https://bluebutton.cms.gov/fhi…","""164660944""",2026-06-16 02:21:57.152,"""C:\BCDA_V3\Data\ExplanationOfB…"


In [51]:
def process_eob_supporting(df_eob: pl.DataFrame):
    
    def safe_expr(df, source_col, expr, alias):
        if source_col in df.columns:
            return expr.alias(alias)
        return pl.lit(None).alias(alias)
    
    column_names = [
            'id',
            'sequence',
            'timingDate',
            'valueString',
            'supportingInfo_category_code',
            'supportingInfo_code',
            'supportingInfo_code_display',
            'supportingInfo_value',
            'supportingInfo_unit',
            'patient_id',
            'lastUpdated',
            'filename',
            'extract_date'
    ]
    
    df_eob_supporting = (
        df_eob
        .filter(pl.col('supportingInfo').is_not_null())
        .select(
            'id',
            'supportingInfo',
            'patient',
            'meta',
            'filename'
        )
        .pipe(flatten, 'supportingInfo')
        .with_columns(
            pl.lit(now).alias('extract_date')
        )
    )
    
    expres = [
        safe_expr(
            df_eob_supporting,
            'valueQuantity',
            pl.col('valueQuantity').struct.field('value'),
            'supportingInfo_value'
        ),
        safe_expr(
            df_eob_supporting,
            'valueQuantity',
            pl.col('valueQuantity').struct.field('unit'),
            'supportingInfo_unit'
        ),
        safe_expr(
            df_eob_supporting,
            'patient',
            pl.col('patient').struct.field('reference').str.split('/').list.get(1),
            'patient_id'
        ),
        safe_expr(
            df_eob_supporting,
            'meta',
            pl.col('meta').struct.field('lastUpdated'),
            'lastUpdated'
        ),
        safe_expr(
            df_eob_supporting,
            'category',
            pl.col('category').struct.field('coding').list.get(0).struct.field('code'),
            'supportingInfo_category_code'
        ),
        safe_expr(
            df_eob_supporting,
            'code',
            pl.col('code').struct.field('coding').list.get(0).struct.field('code'),
            'supportingInfo_code'
        ),
        safe_expr(
            df_eob_supporting,
            'code',
            pl.col('code').struct.field('coding').list.get(0).struct.field('display'),
            'supportingInfo_code_display'
        ),
    ]
    
    df_eob_supporting = (
        df_eob_supporting
        .with_columns(expres)
        .select(column_names)
    )
    
    return df_eob_supporting.to_pandas()

process_eob_supporting(df_eob)

,id,sequence,timingDate,valueString,supportingInfo_category_code,supportingInfo_code,supportingInfo_code_display,supportingInfo_value,supportingInfo_unit,patient_id,lastUpdated,filename,extract_date
0,1004324545686,1,NaT,None,CLM_ADJSTMT_TYPE_CD,0,ORIGINAL,NaN,None,176077134,2026-03-08 09:36:25.384,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 08:17:19
1,1004324545686,2,2024-10-28,None,CLM_IDR_LD_DT,None,None,NaN,None,176077134,2026-03-08 09:36:25.384,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 08:17:19
2,1004324545686,3,NaT,None,RNDRG_PRVDR_FIPS_ST_CD,48,None,NaN,None,176077134,2026-03-08 09:36:25.384,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 08:17:19
3,1004324545686,4,2024-10-21,None,clmrecvddate,None,None,NaN,None,176077134,2026-03-08 09:36:25.384,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 08:17:19
4,1004324545686,5,NaT,None,CLM_CNTRCTR_NUM,04412,"TEXAS - NOVITAS SOLUTIONS, INC. (EFF. 11/17/2012)",NaN,None,176077134,2026-03-08 09:36:25.384,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 08:17:19
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3602969,1006359631211,6,NaT,00000000,CLM_CLNCL_TRIL_NUM,None,None,NaN,None,164660944,2026-06-16 02:21:57.152,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0449_2...,06/07/2026 08:17:19
3602970,1006359631211,7,NaT,None,CLM_DISP_CD,01,None,NaN,None,164660944,2026-06-16 02:21:57.152,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0449_2...,06/07/2026 08:17:19
3602971,1006359631211,8,NaT,None,CLM_QUERY_CD,1,INTERIM BILL,NaN,None,164660944,2026-06-16 02:21:57.152,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0449_2...,06/07/2026 08:17:19
3602972,1006359631211,9,2026-06-12,None,CLM_NCH_WKLY_PROC_DT,None,None,NaN,None,164660944,2026-06-16 02:21:57.152,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0449_2...,06/07/2026 08:17:19


In [ ]:
def process_eob_supporting(df_eob: pl.DataFrame):
    
    df_eob_supporting = (
        df_eob
        .filter(pl.col('supportingInfo').is_not_null())
        .select(
            'id',
            'supportingInfo',
            'patient',
            'meta',
            'filename'
        )
        .pipe(flatten, 'supportingInfo')
        .with_columns(
            pl.col('valueQuantity').struct.field('value').alias('supportingInfo_value'),
            pl.col('valueQuantity').struct.field('unit').alias('supportingInfo_unit'),
            pl.col('patient').struct.field('reference').str.split('/').list.get(1).alias('patient_id'),
            pl.col('meta').struct.field('lastUpdated').alias('lastUpdated'),
            pl.col('category').struct.field('coding').list.get(0).struct.field('code').alias('supportingInfo_category_code'),
            pl.col('code').struct.field('coding').list.get(0).struct.field('code').alias('supportingInfo_code'),
            pl.col('code').struct.field('coding').list.get(0).struct.field('display').alias('supportingInfo_code_display'),
            pl.lit(now).alias('extract_date')
        )
        .select(
            'id',
            'sequence',
            'timingDate',
            'valueString',
            'supportingInfo_category_code',
            'supportingInfo_code',
            'supportingInfo_code_display',
            'supportingInfo_value',
            'supportingInfo_unit',
            'patient_id',
            'lastUpdated',
            'filename',
            'extract_date'
        )
    )
    return df_eob_supporting.to_pandas()

id,sequence,timingDate,valueString,supportingInfo_category_code,supportingInfo_code,supportingInfo_code_display,supportingInfo_value,supportingInfo_unit,patient_id,lastUpdated,filename
str,i64,date,str,str,str,str,i64,str,str,datetime[μs],str
"""1004324545686""",1,null,null,"""CLM_ADJSTMT_TYPE_CD""","""0""","""ORIGINAL""",null,null,"""176077134""",2026-03-08 09:36:25.384,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004324545686""",2,2024-10-28,null,"""CLM_IDR_LD_DT""",null,null,null,null,"""176077134""",2026-03-08 09:36:25.384,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004324545686""",3,null,null,"""RNDRG_PRVDR_FIPS_ST_CD""","""48""",null,null,null,"""176077134""",2026-03-08 09:36:25.384,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004324545686""",4,2024-10-21,null,"""clmrecvddate""",null,null,null,null,"""176077134""",2026-03-08 09:36:25.384,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004324545686""",5,null,null,"""CLM_CNTRCTR_NUM""","""04412""","""TEXAS - NOVITAS SOLUTIONS, INC…",null,null,"""176077134""",2026-03-08 09:36:25.384,"""C:\BCDA_V3\Data\ExplanationOfB…"
…,…,…,…,…,…,…,…,…,…,…,…
"""1006359631211""",6,null,"""00000000""","""CLM_CLNCL_TRIL_NUM""",null,null,null,null,"""164660944""",2026-06-16 02:21:57.152,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1006359631211""",7,null,null,"""CLM_DISP_CD""","""01""",null,null,null,"""164660944""",2026-06-16 02:21:57.152,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1006359631211""",8,null,null,"""CLM_QUERY_CD""","""1""","""INTERIM BILL""",null,null,"""164660944""",2026-06-16 02:21:57.152,"""C:\BCDA_V3\Data\ExplanationOfB…"


In [53]:
def process_eob_careteam(df_eob: pl.DataFrame):
    
    def safe_expr(df, source_col, expr, alias):
        if source_col in df.columns:
            return expr.alias(alias)
        return pl.lit(None).alias(alias)
    
    column_names = [
            'id',
            'sequence',
            'provider_name',
            'npi',
            'taxonomy',
            'specialty_code',
            'role',
            'patient_id',
            'lastUpdated',
            'filename',
            'extract_date'
    ]
    
    df_eob_careteam = (
        df_eob
        .filter(pl.col('careTeam').is_not_null())
        .select(
            'id',
            'careTeam',
            'patient',
            'meta',
            'filename'
        )
        .pipe(flatten, 'careTeam')
        .with_columns(
            pl.lit(now).alias('extract_date')
        )
    )
    
    expres = [
        safe_expr(
            df_eob_careteam,
            'provider',
            pl.col('provider').struct.field('display'),
            'provider_name'
        ),
        safe_expr(
            df_eob_careteam,
            'provider',
            pl.col('provider').struct.field('identifier').struct.field('value'),
            'npi'
        ),
        safe_expr(
            df_eob_careteam,
            'provider',
            pl.col('provider').struct.field('type'),
            'type'
        ),
        safe_expr(
            df_eob_careteam,
            'qualification',
            pl.col('qualification').struct.field('coding').list.get(0).struct.field('code'),
            'specialty_code'
        ),
        safe_expr(
            df_eob_careteam,
            'qualification',
            pl.col('qualification').struct.field('coding').list.get(-1).struct.field('code'),
            'taxonomy'
        ),
        safe_expr(
            df_eob_careteam,
            'role',
            pl.col('role').struct.field('coding').list.get(1).struct.field('code'),
            'role'
        ),
        safe_expr(
            df_eob_careteam,
            'patient',
            pl.col('patient').struct.field('reference').str.split('/').list.get(-1),
            'patient_id'
        ),
        safe_expr(
            df_eob_careteam,
            'meta',
            pl.col('meta').struct.field('lastUpdated'),
            'lastUpdated'
        ),
    ]
    
    df_eob_careteam = (
        df_eob_careteam
        .with_columns(expres)
        .select(column_names)
    )
    
    return df_eob_careteam.to_pandas()

process_eob_careteam(df_eob)

,id,sequence,provider_name,npi,taxonomy,specialty_code,role,patient_id,lastUpdated,filename,extract_date
0,1004324545686,1,"PRILL, CARRIE",1760915664,363L00000X,50,rendering,176077134,2026-03-08 09:36:25.384,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 08:17:19
1,1004357821502,1,"MUKKAVILLI, VENKATA",1992943658,2084P0800X,26,rendering,176077134,2026-03-08 11:46:00.861,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 08:17:19
2,1004357822988,1,V MUKKAV,1992943658,207QS1201X,C0,attending,176077134,2026-03-07 10:43:55.187,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 08:17:19
3,1004377065539,1,MOBILE X-RAY OF AUSTIN INC,1306845961,293D00000X,47,rendering,176077134,2026-03-09 01:19:56.899,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 08:17:19
4,1004377065539,2,"SMALL, CHRISTIAN",1740546209,None,None,referring,176077134,2026-03-09 01:19:56.899,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0006_2...,06/07/2026 08:17:19
...,...,...,...,...,...,...,...,...,...,...,...
434618,1006337971749,2,"GARDIAL, PAUL",1639170194,None,None,referring,164660944,2026-06-09 01:21:00.072,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0449_2...,06/07/2026 08:17:19
434619,1006337972436,1,M RENFRO,1003819913,207T00000X,14,attending,164660944,2026-06-09 01:01:43.513,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0449_2...,06/07/2026 08:17:19
434620,1006337972436,2,M RENFRO,1003819913,207T00000X,14,operating,164660944,2026-06-09 01:01:43.513,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0449_2...,06/07/2026 08:17:19
434621,1006359631211,1,"REULAND, KURT",1063472579,2085R0202X,30,rendering,164660944,2026-06-16 02:21:57.152,C:\BCDA_V3\Data\ExplanationOfBenefit_v3_0449_2...,06/07/2026 08:17:19


In [ ]:
def process_eob_careteam(df_eob: pl.DataFrame):
    df_eob_careteam = (
        df_eob
        .filter(pl.col('careTeam').is_not_null())
        .select(
            'id',
            'careTeam',
            'patient',
            'meta',
            'filename'
        )
        .pipe(flatten, 'careTeam')
        .with_columns(
            pl.col('provider').struct.field('display').alias('provider_name'),
            pl.col('provider').struct.field('identifier').struct.field('value').alias('npi'),
            pl.col('provider').struct.field('type').alias('type'),
            pl.col('qualification').struct.field('coding').list.get(0).struct.field('code').alias('specialty_code'),
            pl.col('qualification').struct.field('coding').list.get(-1).struct.field('code').alias('taxonomy'),
            pl.col('role').struct.field('coding').list.get(1).struct.field('code').alias('role'),
            pl.col('patient').struct.field('reference').str.split('/').list.get(-1).alias('patient_id'),
            pl.col('meta').struct.field('lastUpdated'),
            pl.lit(now).alias('extract_date')
        )
        .select(
            'id',
            'sequence',
            'provider_name',
            'npi',
            'taxonomy',
            'specialty_code',
            'role',
            'patient_id',
            'lastUpdated',
            'filename',
            'extract_date'
        )
    )
    return df_eob_careteam.to_pandas()

id,sequence,provider_name,npi,taxonomy,specialty_code,role,patient_id,lastUpdated,filename
str,i64,str,str,str,str,str,str,datetime[μs],str
"""1004324545686""",1,"""PRILL, CARRIE""","""1760915664""","""363L00000X""","""50""","""rendering""","""176077134""",2026-03-08 09:36:25.384,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004357821502""",1,"""MUKKAVILLI, VENKATA""","""1992943658""","""2084P0800X""","""26""","""rendering""","""176077134""",2026-03-08 11:46:00.861,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004357822988""",1,"""V MUKKAV""","""1992943658""","""207QS1201X""","""C0""","""attending""","""176077134""",2026-03-07 10:43:55.187,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004377065539""",1,"""MOBILE X-RAY OF AUSTIN INC""","""1306845961""","""293D00000X""","""47""","""rendering""","""176077134""",2026-03-09 01:19:56.899,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1004377065539""",2,"""SMALL, CHRISTIAN""","""1740546209""",null,null,"""referring""","""176077134""",2026-03-09 01:19:56.899,"""C:\BCDA_V3\Data\ExplanationOfB…"
…,…,…,…,…,…,…,…,…,…
"""1006337971749""",2,"""GARDIAL, PAUL""","""1639170194""",null,null,"""referring""","""164660944""",2026-06-09 01:21:00.072,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1006337972436""",1,"""M RENFRO""","""1003819913""","""207T00000X""","""14""","""attending""","""164660944""",2026-06-09 01:01:43.513,"""C:\BCDA_V3\Data\ExplanationOfB…"
"""1006337972436""",2,"""M RENFRO""","""1003819913""","""207T00000X""","""14""","""operating""","""164660944""",2026-06-09 01:01:43.513,"""C:\BCDA_V3\Data\ExplanationOfB…"


In [37]:
from datetime import datetime

now = datetime.now().strftime('%d/%m/%Y %H:%M:%S')
now

'02/07/2026 15:28:10'

In [93]:
df_eob

shape: (244_161, 26)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ adjudicat ┆ billableP ┆ careTeam  ┆ contained ┆ … ┆ subType   ┆ related   ┆ procedure ┆ filename │
│ ion       ┆ eriod     ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ ---       ┆ ---       ┆ list[stru ┆ list[stru ┆   ┆ struct[1] ┆ list[stru ┆ list[stru ┆ str      │
│ list[stru ┆ struct[2] ┆ ct[4]]    ┆ ct[11]]   ┆   ┆           ┆ ct[2]]    ┆ ct[4]]    ┆          │
│ ct[4]]    ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ [{{[{"ben ┆ {2024-10- ┆ [{{"PRILL ┆ [{true,"i ┆ … ┆ null      ┆ null      ┆ null      ┆ C:\BCDA_ │
│ efitpayme ┆ 14,2024-1 ┆ , CARRIE" ┆ nsurer-or ┆   ┆           ┆           ┆           ┆ V3\Data\ │
│ ntstatus" ┆ 0-14}     ┆ ,{"http:/ ┆ g",{["htt ┆   ┆           ┆           ┆           ┆ Explanat │
│ ,"B…      ┆           ┆ /hl…      ┆ p:/…      ┆   ┆           ┆           ┆           ┆ ionOfB…  │
│ [{{[{"ben ┆ {2024-09- ┆ [{{"MUKKA ┆ [{true,"i ┆ … ┆ null      ┆ null      ┆ null      ┆ C:\BCDA_ │
│ efitpayme ┆ 05,2024-0 ┆ VILLI,    ┆ nsurer-or ┆   ┆           ┆           ┆           ┆ V3\Data\ │
│ ntstatus" ┆ 9-05}     ┆ VENKATA", ┆ g",{["htt ┆   ┆           ┆           ┆           ┆ Explanat │
│ ,"B…      ┆           ┆ {"htt…    ┆ p:/…      ┆   ┆           ┆           ┆           ┆ ionOfB…  │
│ [{{[{"ben ┆ {2024-09- ┆ [{{"V     ┆ [{true,"i ┆ … ┆ {[{"outpa ┆ null      ┆ null      ┆ C:\BCDA_ │
│ efitpayme ┆ 05,2024-0 ┆ MUKKAV",{ ┆ nsurer-or ┆   ┆ tient","h ┆           ┆           ┆ V3\Data\ │
│ ntstatus" ┆ 9-05}     ┆ "http://h ┆ g",{["htt ┆   ┆ ttp://hl7 ┆           ┆           ┆ Explanat │
│ ,"B…      ┆           ┆ l7.or…    ┆ p:/…      ┆   ┆ .or…      ┆           ┆           ┆ ionOfB…  │
│ [{{[{"ben ┆ {2024-09- ┆ [{{"MOBIL ┆ [{true,"i ┆ … ┆ null      ┆ null      ┆ null      ┆ C:\BCDA_ │
│ efitpayme ┆ 24,2024-0 ┆ E X-RAY   ┆ nsurer-or ┆   ┆           ┆           ┆           ┆ V3\Data\ │
│ ntstatus" ┆ 9-24}     ┆ OF AUSTIN ┆ g",{["htt ┆   ┆           ┆           ┆           ┆ Explanat │
│ ,"B…      ┆           ┆ INC…      ┆ p:/…      ┆   ┆           ┆           ┆           ┆ ionOfB…  │
│ [{{[{"ben ┆ {2024-09- ┆ [{{"J     ┆ [{true,"i ┆ … ┆ {[{"outpa ┆ [{{"https ┆ null      ┆ C:\BCDA_ │
│ efitpayme ┆ 04,2024-0 ┆ CHESTE",{ ┆ nsurer-or ┆   ┆ tient","h ┆ ://bluebu ┆           ┆ V3\Data\ │
│ ntstatus" ┆ 9-04}     ┆ "http://h ┆ g",{["htt ┆   ┆ ttp://hl7 ┆ tton.cms. ┆           ┆ Explanat │
│ ,"B…      ┆           ┆ l7.or…    ┆ p:/…      ┆   ┆ .or…      ┆ gov…      ┆           ┆ ionOfB…  │
│ …         ┆ …         ┆ …         ┆ …         ┆ … ┆ …         ┆ …         ┆ …         ┆ …        │
│ [{{[{"ben ┆ {2026-05- ┆ [{{"J     ┆ [{true,"i ┆ … ┆ {[{"outpa ┆ null      ┆ null      ┆ C:\BCDA_ │
│ efitpayme ┆ 27,2026-0 ┆ FERGUS",{ ┆ nsurer-or ┆   ┆ tient","h ┆           ┆           ┆ V3\Data\ │
│ ntstatus" ┆ 5-27}     ┆ "http://h ┆ g",{["htt ┆   ┆ ttp://hl7 ┆           ┆           ┆ Explanat │
│ ,"B…      ┆           ┆ l7.or…    ┆ p:/…      ┆   ┆ .or…      ┆           ┆           ┆ ionOfB…  │
│ [{{[{"ben ┆ {2026-05- ┆ [{{"MILLS ┆ [{true,"i ┆ … ┆ null      ┆ null      ┆ null      ┆ C:\BCDA_ │
│ efitpayme ┆ 26,2026-0 ┆ , BRITTNY ┆ nsurer-or ┆   ┆           ┆           ┆           ┆ V3\Data\ │
│ ntstatus" ┆ 5-26}     ┆ ",{"http: ┆ g",{["htt ┆   ┆           ┆           ┆           ┆ Explanat │
│ ,"B…      ┆           ┆ //h…      ┆ p:/…      ┆   ┆           ┆           ┆           ┆ ionOfB…  │
│ [{{[{"ben ┆ {2026-05- ┆ [{{"CARIM ┆ [{true,"i ┆ … ┆ null      ┆ null      ┆ null      ┆ C:\BCDA_ │
│ efitpayme ┆ 27,2026-0 ┆ I, KYLIE" ┆ nsurer-or ┆   ┆           ┆           ┆           ┆ V3\Data\ │
│ ntstatus" ┆ 5-27}     ┆ ,{"http:/ ┆ g",{["htt ┆   ┆           ┆           ┆           ┆ Explanat │
│ ,"B…      ┆           ┆ /hl…      ┆ p:

In [ ]:
def process_eob_item(df_eob: pl.DataFrame):
    column_names = [
            'id',
            'sequence',
            'diagnosisSequence',
            'informationSequence',
            'location_code',
            'location_display',
            'product_service_code',
            'modifier',
            'product_service_system',
            'quantity_value',
            'revenue_code',
            'revenue_display',
            'service_start',
            'service_end',
            'servicedDate',
            'patient_id',
            'lastUpdated',
            'filename',
            'extract_date'
    ]
    
    df_eob_item = (
        df_eob
        .filter(pl.col('item').is_not_null())
        .select(
            'id',
            'item',
            'patient',
            'meta',
            'filename'
        )
        .pipe(flatten, 'item')
        .with_columns(
            pl.col('productOrService').struct.field('coding').list.get(0).struct.field('code').alias('product_service_code'),
            pl.col('productOrService').struct.field('coding').list.get(0).struct.field('system').str.split('/').list.get(-1).alias('product_service_system'),
            pl.col('quantity').struct.field('value').alias('quantity_value'),
            pl.col('patient').struct.field('reference').str.split('/').list.get(1).alias('patient_id'),
            pl.col('meta').struct.field('lastUpdated').alias('lastUpdated'),
            pl.col('revenue').struct.field('coding').list.get(0).struct.field('code').alias('revenue_code'),
            pl.col('revenue').struct.field('coding').list.get(0).struct.field('display').alias('revenue_display'),
            pl.col('servicedPeriod').struct.field('start').alias('service_start'),
            pl.col('servicedPeriod').struct.field('end').alias('service_end'),
            pl.col('locationCodeableConcept').struct.field('coding').list.get(0).struct.field('code').alias('location_code'),
            pl.col('locationCodeableConcept').struct.field('coding').list.get(0).struct.field('display').alias('location_display'),
            pl.col('diagnosisSequence').list.get(0).alias('diagnosisSequence'),
            pl.col('informationSequence').list.get(0).alias('informationSequence'),
            pl.col('modifier').list.get(0).struct.field('coding').list.get(0).struct.field('code').alias('modifier'),
            pl.lit(now).alias('extract_date'),
            pl.when(pl.col('servicedDate') < MIN_SQL_DATE)
                .then(None)
                .otherwise(pl.col('servicedDate'))
                .alias('servicedDate')
                
        )
        .select(column_names)
    )
    return df_eob_item.to_pandas()

In [144]:
import fastexcel

df = pl.read_csv(r'C:\BCDA_V3\v3-data-dictionary-2.248.0.csv')
urls = df.filter(pl.col('referenceTable').is_not_null()).get_column('referenceTable').to_list()
urls

['https://bluebutton.cms.gov/fhir/CodeSystem/CLM-TYPE-CD',
 'https://bluebutton.cms.gov/fhir/CodeSystem/CLM-SRC-ID',
 'https://bluebutton.cms.gov/fhir/CodeSystem/CLM-SBMT-FRMT-CD',
 'https://bluebutton.cms.gov/fhir/CodeSystem/CLM-PHRMCY-SRVC-TYPE-CD',
 'https://bluebutton.cms.gov/fhir/CodeSystem/CLM-LINE-RX-ORGN-CD',
 'https://bluebutton.cms.gov/fhir/CodeSystem/CLM-BRND-GNRC-CD',
 'https://bluebutton.cms.gov/fhir/CodeSystem/CLM-PTNT-RSDNC-CD',
 'https://bluebutton.cms.gov/fhir/CodeSystem/CLM-LTC-DSPNSNG-MTHD-CD',
 'https://bluebutton.cms.gov/fhir/CodeSystem/CLM-CMPND-CD',
 'https://bluebutton.cms.gov/fhir/CodeSystem/CLM-DRUG-CVRG-STUS-CD',
 'https://bluebutton.cms.gov/fhir/CodeSystem/CLM-CTSTRPHC-CVRG-IND-CD',
 'https://bluebutton.cms.gov/fhir/CodeSystem/CLM-ADJSTMT-TYPE-CD',
 'https://bluebutton.cms.gov/fhir/CodeSystem/CLM-DSPNSNG-STUS-CD',
 'https://bluebutton.cms.gov/fhir/CodeSystem/Final-Action',
 'https://bluebutton.cms.gov/fhir/CodeSystem/CLM-TYPE-CD',
 'https://bluebutton.cms.go

In [116]:
print(urls[0])

shape: (1, 1)
┌─────────────────────────────────┐
│ referenceTable                  │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ https://bluebutton.cms.gov/fhi… │
└─────────────────────────────────┘


In [122]:
df = pl.DataFrame({
    'value': pl.String,
    'description': pl.String,
    'url' : pl.String
})
df

value,description,url
object,object,object
String,String,String


In [136]:
urls_lst = urls.to_list()
urls_lst

AttributeError: 'DataFrame' object has no attribute 'to_list'

In [134]:
df_url = pd.read_html('https://bluebutton.cms.gov/fhir/CodeSystem/CLM-TYPE-CD')
df_url[0]['Value']

0         1
1         2
2         3
3         4
4        10
       ... 
119    2098
120    2099
121    2700
122    2800
123    2900
Name: Value, Length: 124, dtype: int64

In [152]:

rows = []

for url in urls:
    if requests.get(url).status_code != 200:
        continue

    table = pd.read_html(url)[0]

    rows.extend(
        {
            "value": value,
            "description": description,
            "url": url,
        }
        for value, description in zip(
            table["Value"],
            table["Description"]
        )
    )

df = pd.DataFrame(rows)

In [154]:
from Credentials import engine_DEV_Test as engine

df.to_sql('blueButton_LookupTable', engine, index= False)

-1

In [146]:
rows = []


for url in urls:
    if requests.get(url).status_code != 200:
        continue
    df_url = pd.read_html(url)
    rows.append({
        'values': df_url[0]['Value'],
        'description': df_url[0]['Description'],
        'url': url
        })
df = pl.DataFrame(rows)
df
#df_url


values,description,url
object,object,str
"0 1 1 2 2 3 3 4 4 10 ... 119 2098 120 2099 121 2700 122 2800 123 2900 Name: Value, Length: 124, dtype: int64","0 MEDICARE PART D ORIGINAL CLAIM 1 MEDICARE PART D ADJUSTED CLAIM 2 MEDICARE PART D DELETED CLAIM 3 MEDICARE PART D RESUBMITTED CLAIM 4 MEDICARE HHA CLAIM ... 119 098X MEDICARE SS RESERVED FOR NATIONAL ASSIGNMENT 120 099X MEDICARE SS RESERVED FOR NATIONAL ASSIGNMENT 121 PROF MEDICARE SS PART B PROFESSIONAL 122 DME MEDICARE SS DME 123 HOSPICE NOTICE OF ELECTION Name: Description, Length: 124, dtype: object","""https://bluebutton.cms.gov/fhi…"
"0 20000 1 21000 2 22000 3 23000 Name: Value, dtype: int64","0 National Claims History (NCH) 1 Fiscal Intermediary Shared System (FISS) 2 Multi-Carrier System (MCS) 3 Viable Information Processing Systems (ViPS) M... Name: Description, dtype: object","""https://bluebutton.cms.gov/fhi…"
"0 S 1 P 2 X 3 C 4 B 5 N 6 A Name: Value, dtype: object","0 STATE-TO-PLAN PDES 1 PAPER CLAIM FROM PROVIDER 2 X12 837 3 COB CLAIM 4 BENEFICIARY SUBMITTED 5 NCPDP ELECTRONIC SUBMISSION 6 MEDICAID SUBROGATION CLAIM Name: Description, dtype: object","""https://bluebutton.cms.gov/fhi…"
"0 1 1 2 2 3 3 4 4 5 5 6 6 7 7 8 8 99 Name: Value, dtype: int64","0 Community/retail pharmacy 1 Compounding pharmacy 2 Home infusion therapy provider 3 Institutional pharmacy 4 Long-term care pharmacy 5 Mail order pharmacy 6 Managed care organization (MCO) pharmacy 7 Specialty care pharmacy 8 Other Name: Description, dtype: object","""https://bluebutton.cms.gov/fhi…"
"0 0 1 1 2 2 3 3 4 4 5 5 Name: Value, dtype: int64","0 Not specified 1 Written 2 Telephone 3 Electronic 4 Facsimile 5 Pharmacy Name: Description, dtype: object","""https://bluebutton.cms.gov/fhi…"
…,…,…
"0 N 1 F 2 U 3 P Name: Value, dtype: object","0 NONE 1 FULL DUAL 2 UNKNOWN 3 PARTIAL DUAL Name: Description, dtype: object","""https://bluebutton.cms.gov/fhi…"
"0 10 1 11 2 12 3 13 4 14 5 3 6 4 7 5 8 6 9 7 10 8 11 9 12 NF 13 U 14 ~ Name: Value, dtype: object","0 CHCCP 1 PDP 2 CCDM 3 MSADM 4 MMP 5 CCP 6 MSA 7 PFFS 8 PACE 9 PCE 10 DEMO 11 FFS 12 NF 13 UNK 14 NO DESCRIPTION AVAILABLE Name: Description, dtype: object","""https://bluebutton.cms.gov/fhi…"
"0 Y 1 N 2 U Name: Value, dtype: object","0 Yes 1 No 2 Unknown Name: Description, dtype: object","""https://bluebutton.cms.gov/fhi…"


In [ ]:
import requests
from bs4 import BeautifulSoup

response = requests.get(url='https://bluebutton.cms.gov/fhir/CodeSystem/CLM-TYPE-CD')
response.raise_for_status()



soup = BeautifulSoup(response.text, 'html.parser')
soup

<!DOCTYPE html>
<html class="usa-js-loading" lang="en"> <head><title>Variable: Claim Type Code - CMS Blue Button API</title><meta charset="utf-8"/><link href="https://bluebutton.cms.gov/fhir/CodeSystem/CLM-TYPE-CD/" rel="canonical"/><meta content="CA CODE IDENTIFYING THE SOURCE AND TYPE OF CLAIM SUBMITTED THROUGH THE MEDICARE OR MEDICAID PROGRAM." name="description"/><meta content="index, follow" name="robots"/><meta content="Variable: Claim Type Code" property="og:title"/><meta content="Website" property="og:type"/><meta content="https://bluebutton.cms.gov/meta/og.jpg" property="og:image"/><meta content="https://bluebutton.cms.gov/fhir/CodeSystem/CLM-TYPE-CD/" property="og:url"/><meta content="CA CODE IDENTIFYING THE SOURCE AND TYPE OF CLAIM SUBMITTED THROUGH THE MEDICARE OR MEDICAID PROGRAM." property="og:description"/><meta content="en_US" property="og:locale"/><meta content="Blue Button" property="og:site_name"/><meta content="https://bluebutton.cms.gov/meta/og.jpg" property="og:im

In [170]:
html = """PGRpdj48dWw+PGxpIGlkPSJIaXN0b3J5QW5kUGh5c2ljYWxOb3RlMSI+PHBhcmFncmFwaD5IUEkgKEhpc3Rvcnkgb2YgUHJlc2VudCBJbGxuZXNzKTwvcGFyYWdyYXBoPjx0YWJsZSB3aWR0aD0iMTAwJSIgYm9yZGVyPSIxIj48Y29sZ3JvdXA+PGNvbCB3aWR0aD0iMTAlIi8+PGNvbCB3aWR0aD0iMTAlIi8+PGNvbCB3aWR0aD0iMTUlIi8+PGNvbCB3aWR0aD0iMTUlIi8+PGNvbCB3aWR0aD0iMTUlIi8+PGNvbCB3aWR0aD0iMTUlIi8+PGNvbCB3aWR0aD0i
MjAlIi8+PC9jb2xncm91cD48dGhlYWQ+PHRyPjx0aD5DYXRlZ29yeTwvdGg+PHRoPmMvbzwvdGg+PHRoPkRlbmllczwvdGg+PHRoPlN5bXB0b208L3RoPjx0aD5EdXJhdGlvbjwvdGg+PHRoPkRldGFpbHM8L3RoPjx0aD5Ob3RlczwvdGg+PHRoPkNhdGVnb3J5IE5vdGVzPC90aD48L3RyPjwvdGhlYWQ+PHRib2R5Pjx0cj48dGQgcm93c3Bhbj0iNiI+RnVu
Y3Rpb25hbCBTdGF0dXM8L3RkPjx0ZCByb3dzcGFuPSIzIi8+PHRkIHJvd3NwYW49IjMiLz48dGQgcm93c3Bhbj0iMyI+RnVuY3Rpb25hbCBDb2duaXRpdmUgQXNzZXNzbWVudDo8L3RkPjx0ZCByb3dzcGFuPSIzIi8+PHRkPkZ1bmN0aW9uYWwgQ29nbml0aXZlIEFzc2Vzc21lbnQgMTo6IFN0YXR1cyAxPC90ZD48dGQ+Tm90ZXNAMTIzITwvdGQ+PHRkIHJvd3NwYW49IjYiLz48L3RyPjx0cj48dGQ+RnVuY3Rpb25hbCBDb2duaXRpdmUgQXNzZXNzbWVudCA4OjogMzwvdGQ+PHRkPk5vdGVzQDEyMyE8L3RkPjwvdHI+PHRyPjx0ZD5GdW5jdGlvbmFsIENvZ25pdGl2ZSBBc3Nlc3NtZW50IDExOjogMDEvMDEvMjAyNTwvdGQ+PHRkPk5vdGVzQDEyMyE8L3RkPjwvdHI+PHRyPjx0ZCByb3dzcGFuPSIzIi8+PHRkIHJvd3NwYW49IjMiLz48dGQgcm93c3Bhbj0iMyI+Q29nbml0aXZlIEFzc2Vzc21lbnQ6PC90ZD48dGQgcm93c3Bhbj0iMyIvPjx0ZD5Db2duaXR
pdmUgU3RhdHVzIDQ6IFN0YXR1cyA0PC90ZD48dGQ+Tm90ZXNAMTIzITwvdGQ+PC90cj48dHI+PHRkPkNvZ25pdGl2ZSBTdGF0dXMgMTA6IDQzLjc8L3RkPjx0ZD5Ob3Rlc0AxMjMhPC90ZD48L3RyPjx0cj48dGQ+Q29nbml0aXZlIFN0YXR1cyAxMTogMDEvMDEvMjAyNTwvdGQ+PHRkPk5vdGVzQDEyMyE8L3RkPjwvdHI+PC90Ym9keT48L3RhYmxlPjxicj48L2JyPjwvbGk+PC91bD48L2Rpdj4="""

In [177]:
from io import StringIO
import base64



soup = BeautifulSoup(html, 'lxml')

encoded = soup.find('p').text.strip()

html_2 = base64.b64decode(encoded).decode("utf-8")


#print(html)
df = pd.read_html(StringIO(html_2))[0]
df

,Category,c/o,Denies,Symptom,Duration,Details,Notes,Category Notes
0,Functional Status,NaN,NaN,Functional Cognitive Assessment:,NaN,Functional Cognitive Assessment 1:: Status 1,Notes@123!,NaN
1,Functional Status,NaN,NaN,Functional Cognitive Assessment:,NaN,Functional Cognitive Assessment 8:: 3,Notes@123!,NaN
2,Functional Status,NaN,NaN,Functional Cognitive Assessment:,NaN,Functional Cognitive Assessment 11:: 01/01/2025,Notes@123!,NaN
3,Functional Status,NaN,NaN,Cognitive Assessment:,NaN,Cognitive Status 4: Status 4,Notes@123!,NaN
4,Functional Status,NaN,NaN,Cognitive Assessment:,NaN,Cognitive Status 10: 43.7,Notes@123!,NaN
5,Functional Status,NaN,NaN,Cognitive Assessment:,NaN,Cognitive Status 11: 01/01/2025,Notes@123!,NaN
